# SeamlessM4T v2 → Textless Pure S2ST (~673M)
## Voice Cloning · 5 Languages · Long-Form Audio · INTERSPEECH/IWSLT 2026

**Architectural transformation paper** — converting SeamlessM4T v2 from text-mediated S2ST
to a fully textless, speaker-preserving, long-form-capable S2ST system.

| Component | Original | After |
|---|---|---|
| Text Decoder | 867M, 24 layers | **0M — permanently removed** |
| lm_head + shared vocab | ~262M | **0M — removed** |
| Speech Encoder | 635M, 24L | **~441M, 16L** (BI+iterative prune) |
| T2U Model | 262M, 6+6L | **~175M, 4+4L** (LaCo RDSC merge) |
| CIF Connector | — | **~5M NEW** (trained from scratch) |
| Speaker Adapter | — | **~0.1M NEW** (ECAPA→vocoder 192→256) |
| **Total** | **1805M** | **~673M** |

**Papers:** S2UT (Lee ACL 2022) · SeamlessExpressive (arXiv:2312.05187) ·
LaCo (Yang EMNLP 2024) · CIF (Dong & Xu ICASSP 2020) · ECAPA-TDNN (Desplanques IS 2020) ·
DoRA (Liu ICML 2024) · ShortGPT (ACL 2025) · MMS (Pratap 2023)

**Phases:** P0 V1-Baseline → P1 Vocab5L → P2 EncPrune16L → P3 LaCoT2U →
P4 TextlessArch → P5 KD-Extract → P6a CIF-FeatureKD → P6b E2E-DoRA → P7 FullBenchmark


## ⚙️ Setup — run ALL at the start of EVERY Kaggle session

In [ ]:
import os, sys, subprocess, pathlib, re, glob, json, gc, copy, time, math, shutil, random
import warnings; warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = not ON_KAGGLE
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

GDRIVE_MOUNT = '/content/drive/MyDrive/seamTL'   # ← NEW project folder
KAGGLE_WORK  = '/kaggle/working'

WORK_DIR  = KAGGLE_WORK if ON_KAGGLE else GDRIVE_MOUNT
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'
FIG_DIR   = f'{WORK_DIR}/figures'
MODEL_DIR = f'{WORK_DIR}/models'

GDRIVE_ROOT = 'gdrive:seamTL'   # rclone remote root (Kaggle only)


In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Drive mounted. Working folder: {GDRIVE_MOUNT}')
else:
    print('Kaggle: skipping Drive mount.')


In [ ]:
for d in [WORK_DIR, CKPT_DIR, AUDIO_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Platform : {PLATFORM}')
print(f'Work dir : {WORK_DIR}')
print(f'Checkpts : {CKPT_DIR}')


In [ ]:
if ON_KAGGLE:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash',
                   shell=True, capture_output=True)
    ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
    print(ver.stdout.split('\n')[0])
else:
    print('Colab: rclone not needed — using mounted Drive directly.')
    if not os.path.exists('/content/drive/MyDrive'):
        print('WARNING: Drive does not appear to be mounted.')
    else:
        print('Drive mount: OK')


In [ ]:
def _get_secret(key):
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key)
        except Exception as e:
            raise RuntimeError(f'Kaggle secret {key!r} not found: {e}')
    else:
        try:
            from google.colab import userdata
            return userdata.get(key)
        except Exception as e:
            raise RuntimeError(f'Colab secret {key!r} not found: {e}')

if ON_KAGGLE:
    RCLONE_CONF = _get_secret('RCLONE_CONF')
    raw = RCLONE_CONF.strip()
    raw = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', raw)
    raw = re.sub(r'\s+(type|scope|token|team_drive|client_id|client_secret|'
                 r'root_folder_id|service_account_file|drive_id)\s*=\s*',
                 r'\n\1 = ', raw)
    raw = raw.strip() + '\n'
    rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
    rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
    rclone_cfg.write_text(raw)
    r = subprocess.run('rclone lsd gdrive:', shell=True, capture_output=True, text=True)
    print('Drive root:' if r.returncode == 0 else 'rclone FAILED:')
    print(r.stdout[:300] or r.stderr[:300])
else:
    print('Colab: skipping rclone config.')

try:
    HF_TOKEN = _get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(HF_TOKEN)
    print('HuggingFace login: OK')
except Exception as e:
    print(f'HF login skipped: {e}')


In [ ]:
subprocess.run([
    'pip', 'install', '-q',
    'transformers>=4.41.0', 'datasets', 'torchaudio', 'speechbrain>=1.0.0',
    'peft>=0.10.0', 'librosa', 'jiwer', 'evaluate', 'sacrebleu', 'pyarrow',
    'sentencepiece', 'accelerate', 'matplotlib', 'seaborn',
    'soundfile', 'requests', 'pandas',
], check=True)
print('All packages installed.')

In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
import torch
import random
import numpy as np

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
# ── pulled verbatim from seamless-cse465v5 Cell 14 ──
import torch
from datetime import datetime

_CUSTOM_STATE_FILE = '_custom_state.pt'
_PRUNING_MANIFEST  = 'pruning_manifest.pt'

def _rclone_push(local_path, remote_subpath):
    if not ON_KAGGLE: return
    r = subprocess.run(
        f'rclone copy "{local_path}" "{GDRIVE_ROOT}/{remote_subpath}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[rclone] WARNING: push failed for {local_path}: {r.stderr[:200]}')

def _rclone_pull_model(stage_name):
    if not ON_KAGGLE: return
    local = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(local, exist_ok=True)
    r = subprocess.run(
        f'rclone sync "{GDRIVE_ROOT}/models/{stage_name}/" "{local}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'[rclone] model pull failed for {stage_name}: {r.stderr[:300]}')
    print(f'[rclone] Pulled {stage_name} → {local}')

def save_checkpoint(state, name, step=0, keep=3):
    fname = f'{name}_step{step:06d}.pt'
    path  = f'{CKPT_DIR}/{fname}'
    torch.save(state, path)
    mb = os.path.getsize(path) / 1e6
    print(f'[ckpt] Saved {fname} ({mb:.1f} MB)')
    if ON_KAGGLE: _rclone_push(path, 'checkpoints')
    old = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    for f in old[:-keep]:
        if os.path.exists(f): os.remove(f)

def load_latest_checkpoint(name):
    files = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    if not files:
        print(f'[ckpt] No checkpoint for {name!r}')
        return None
    state = torch.load(files[-1], map_location='cpu', weights_only=False)
    print(f'[ckpt] Loaded {os.path.basename(files[-1])}')
    return state

def sync_checkpoints_from_drive():
    if ON_KAGGLE:
        print('[ckpt] Syncing from rclone remote...')
        r = subprocess.run(
            f'rclone sync "{GDRIVE_ROOT}/checkpoints/" "{CKPT_DIR}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
            shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[ckpt] WARNING: {r.stderr[:300]}')
    else:
        print(f'[ckpt] Colab: reading directly from {CKPT_DIR}')
    files = sorted(os.listdir(CKPT_DIR)) if os.path.exists(CKPT_DIR) else []
    print(f'[ckpt] {len(files)} file(s) available')
    for f in files:
        mb = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
        print(f'  {f:<55} {mb:>7.1f} MB')

print('Checkpoint helpers ready.')


In [ ]:
import torch.nn as nn, torch.nn.functional as F

_CUSTOM_ATTR_NAMES = ['_vocab_remap_to_old']

def _save_custom_state(mdl, path):
    state = {a: getattr(mdl, a) for a in _CUSTOM_ATTR_NAMES if hasattr(mdl, a)}
    if state:
        torch.save(state, os.path.join(path, _CUSTOM_STATE_FILE))
        print(f'  Saved custom state: {list(state.keys())}')

def _load_custom_state(mdl, path):
    fpath = os.path.join(path, _CUSTOM_STATE_FILE)
    if not os.path.exists(fpath): return
    state = torch.load(fpath, map_location='cpu', weights_only=False)
    for k, v in state.items(): setattr(mdl, k, v)
    print(f'  Restored custom state: {list(state.keys())}')

def _find_layers(component):
    for attr in ['layers', 'inner_layers', 'layer']:
        mod = getattr(component, attr, None)
        if isinstance(mod, nn.ModuleList) and len(mod) > 0:
            return mod
    return None

def _get_t2u_encoder_decoder(mdl):
    t2u   = getattr(mdl, 't2u_model', None)
    if t2u is None: return None, None
    inner = getattr(t2u, 'model', None)
    if inner is None: return None, None
    return getattr(inner, 'encoder', None), getattr(inner, 'decoder', None)

def sync_model_config(mdl):
    """Keep config in sync with actual ModuleList depths after pruning."""    
    cfg = mdl.config
    if hasattr(mdl, 'speech_encoder'):
        enc = mdl.speech_encoder
        parent = enc.encoder if hasattr(enc, 'encoder') else enc
        if hasattr(parent, 'layers'):
            actual = len(parent.layers)
            for k in ['speech_encoder_layers']:
                if hasattr(cfg, k) and getattr(cfg, k) != actual:
                    print(f'  [config] {k}: {getattr(cfg,k)} -> {actual}')
                    setattr(cfg, k, actual)
            sc = getattr(mdl.speech_encoder, 'config', None)
            if sc and hasattr(sc, 'num_hidden_layers') and sc.num_hidden_layers != actual:
                sc.num_hidden_layers = actual
    if hasattr(mdl, 'text_decoder') and mdl.text_decoder is not None:
        layers = _find_layers(mdl.text_decoder)
        if layers is not None:
            actual = len(layers)
            if hasattr(cfg, 'decoder_layers') and cfg.decoder_layers != actual:
                print(f'  [config] decoder_layers: {cfg.decoder_layers} -> {actual}')
                cfg.decoder_layers = actual
    t2u_enc, t2u_dec = _get_t2u_encoder_decoder(mdl)
    for sub, attr in [(t2u_enc,'t2u_encoder_layers'), (t2u_dec,'t2u_decoder_layers')]:
        if sub is None: continue
        layers = _find_layers(sub)
        if layers and hasattr(cfg, attr) and getattr(cfg, attr) != len(layers):
            print(f'  [config] {attr}: {getattr(cfg,attr)} -> {len(layers)}')
            setattr(cfg, attr, len(layers))
    t2u = getattr(mdl, 't2u_model', None)
    if t2u and hasattr(t2u, 'config'):
        tc = t2u.config
        for sub, attr in [(t2u_enc,'encoder_layers'), (t2u_dec,'decoder_layers')]:
            if sub is None: continue
            layers = _find_layers(sub)
            if layers and hasattr(tc, attr) and getattr(tc, attr) != len(layers):
                print(f'  [config] t2u.config.{attr}: {getattr(tc,attr)} -> {len(layers)}')
                setattr(tc, attr, len(layers))
    print('  [config] sync done.')

def _consolidate_to_single_gpu(mdl):
    """Move model to cuda:0 if split by device_map='auto'."""    
    if not torch.cuda.is_available(): return mdl
    if not (hasattr(mdl, 'hf_device_map') and len(set(mdl.hf_device_map.values())) > 1):
        return mdl
    print('  Multi-device → consolidating to cuda:0...')
    try:
        from accelerate.hooks import remove_hook_from_submodules
        remove_hook_from_submodules(mdl)
    except Exception: pass
    mdl = mdl.to('cuda:0')
    torch.cuda.empty_cache()
    print(f'  Model now on: {next(mdl.parameters()).device}')
    return mdl

def load_hf_weights_dict(model_dir):
    from pathlib import Path
    safe = Path(model_dir) / 'model.safetensors'
    if safe.is_file():
        try:
            from safetensors.torch import load_file
            return load_file(str(safe))
        except ImportError: pass
    pt = Path(model_dir) / 'pytorch_model.bin'
    if pt.is_file():
        blob = torch.load(str(pt), map_location='cpu', weights_only=False)
        return blob.get('model', blob) if isinstance(blob, dict) else blob
    return None

def _infer_t2u_layer_counts(model_dir):
    sd = load_hf_weights_dict(model_dir)
    if not sd: return None, None
    enc_idx, dec_idx = set(), set()
    for k in sd:
        if k.startswith('t2u_model.model.encoder.layers.'):
            r = k.split('.')[4]
            if r.isdigit(): enc_idx.add(int(r))
        elif k.startswith('t2u_model.model.decoder.layers.'):
            r = k.split('.')[4]
            if r.isdigit(): dec_idx.add(int(r))
    return (max(enc_idx)+1 if enc_idx else None), (max(dec_idx)+1 if dec_idx else None)

def save_model_to_drive(mdl, proc, stage_name, manifest_extra=None):
    """Save model to Drive using HF save_pretrained (battle-tested from v5)."""    
    target = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(target, exist_ok=True)
    print(f'[model] Saving {stage_name} → {target} ...')
    sync_model_config(mdl)
    _save_custom_state(mdl, target)
    man = {'stage_name': stage_name}
    if manifest_extra: man.update(manifest_extra)
    torch.save(man, os.path.join(target, _PRUNING_MANIFEST))
    try:
        mdl.save_pretrained(target, safe_serialization=True)
    except Exception as e:
        print(f'  safe_serialization failed ({e}); trying .bin')
        mdl.save_pretrained(target)
    if proc is not None: proc.save_pretrained(target)
    total = sum(os.path.getsize(f'{target}/{f}') for f in os.listdir(target)) / 1e6
    print(f'[model] Local: {total:.0f} MB in {len(os.listdir(target))} files.')
    if ON_KAGGLE:
        r = subprocess.run(f'rclone sync "{target}/" "{GDRIVE_ROOT}/models/{stage_name}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[model] WARNING rclone push failed: {r.stderr[:300]}')
        else:
            print(f'[model] Pushed to remote: {GDRIVE_ROOT}/models/{stage_name}/')
    else:
        print('[model] Colab: saved directly to Drive.')


def save_textless_model_to_drive(mdl, proc, stage_name):
    """
    Save textless model (Phase 4+) that has cif_connector + speaker_adapter.
    Uses state_dict saving instead of save_pretrained because HF doesn't know
    about our custom modules. load_textless_model_from_drive will load it correctly
    via surgery + load_state_dict(strict=False).
    """
    target = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(target, exist_ok=True)
    print(f'[model] Saving textless model {stage_name} → {target} ...')

    sync_model_config(mdl)
    _save_custom_state(mdl, target)

    # Save manifest with architecture metadata
    manifest = {
        'stage_name': stage_name,
        'hidden':     mdl.config.hidden_size,
        'n_langs':    getattr(mdl.config, 'vocoder_num_langs',
                      getattr(mdl.config, 't2u_num_langs', 36)),
        'has_cif':    hasattr(mdl, 'cif_connector'),
        'has_speaker': hasattr(mdl, 'speaker_adapter'),
    }
    torch.save(manifest, os.path.join(target, _PRUNING_MANIFEST))

    # Save full state dict — includes cif_connector + speaker_adapter weights
    # load_textless_model_from_drive rebuilds architecture via surgery then
    # calls load_state_dict(strict=False), so this loads correctly
    sd_path = os.path.join(target, 'pytorch_model.bin')
    torch.save(mdl.state_dict(), sd_path)
    print(f'  Saved state dict: {os.path.getsize(sd_path)/1e6:.0f} MB')

    if proc is not None:
        proc.save_pretrained(target)
        print('  Saved processor.')

    total = sum(os.path.getsize(f'{target}/{f}')
                for f in os.listdir(target)) / 1e6
    print(f'[model] Local: {total:.0f} MB in {len(os.listdir(target))} files.')

    if ON_KAGGLE:
        r = subprocess.run(
            f'rclone sync "{target}/" "{GDRIVE_ROOT}/models/{stage_name}/" '
            f'--transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
            shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[model] WARNING rclone push failed: {r.stderr[:300]}')
        else:
            print(f'[model] Pushed to remote: {GDRIVE_ROOT}/models/{stage_name}/')

    print(f'[model] ✓ {stage_name} saved.')

def load_model_from_drive(stage_name):
    from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor, AutoConfig
    local = f'{MODEL_DIR}/{stage_name}'
    if ON_KAGGLE and (not os.path.exists(local) or not os.listdir(local)):
        print(f'[model] Not in local cache — pulling from remote...')
        _rclone_pull_model(stage_name)
    if not os.path.exists(local) or not os.listdir(local):
        raise RuntimeError(f'[model] Not found or empty: {local}')
    wf = [f for f in os.listdir(local) if f.endswith('.safetensors') or f.endswith('.bin')]
    if not wf:
        raise RuntimeError(f'[model] No weight files in {local}')
    print(f'[model] Loading {stage_name} from {local} ...')
    cfg = AutoConfig.from_pretrained(local)
    enc_n, dec_n = _infer_t2u_layer_counts(local)
    if enc_n and getattr(cfg,'t2u_encoder_layers',None) != enc_n:
        print(f'  Repair T2U enc depth: {cfg.t2u_encoder_layers} -> {enc_n}')
        cfg.t2u_encoder_layers = enc_n
    if dec_n and getattr(cfg,'t2u_decoder_layers',None) != dec_n:
        print(f'  Repair T2U dec depth: {cfg.t2u_decoder_layers} -> {dec_n}')
        cfg.t2u_decoder_layers = dec_n
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        local, config=cfg, torch_dtype=torch.float16, device_map='auto')
    _load_custom_state(mdl, local)
    proc = SeamlessM4TProcessor.from_pretrained(local)
    mdl.eval()
    print(f'[model] Loaded {stage_name}.')
    return mdl, proc

print('Model I/O helpers ready.')


In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib, seaborn as sns
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120, 'savefig.bbox': 'tight'})
sns.set_style('whitegrid')

N_GPU = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs {N_GPU}')
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU{i}: {torch.cuda.get_device_name(i)}  {p.total_memory/1e9:.1f} GB')

def count_params(module):
    return sum(p.numel() for p in module.parameters()) / 1e6

def count_params_detailed(model):
    bd = {n: count_params(c) for n, c in model.named_children()}
    bd['TOTAL'] = count_params(model)
    return bd

def print_model_breakdown(model, title='Model Breakdown'):
    bd = count_params_detailed(model)
    print(f'\n--- {title} ---')
    total = bd.pop('TOTAL')
    for name, p in sorted(bd.items(), key=lambda x: -x[1]):
        pct = p / total * 100 if total > 0 else 0
        print(f'  {name:<35} {p:>8.1f}M  ({pct:>5.1f}%)')
    print(f'  {"TOTAL":<35} {total:>8.1f}M')
    print('---')
    return {**bd, 'TOTAL': total}

def gpu_mem():
    if torch.cuda.is_available():
        for i in range(N_GPU):
            a = torch.cuda.memory_allocated(i)/1e9
            r = torch.cuda.memory_reserved(i)/1e9
            print(f'  GPU{i}: {a:.2f}GB alloc / {r:.2f}GB reserved')

def save_figure(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}', dpi=150, bbox_inches='tight')
    if ON_KAGGLE: _rclone_push(f'{FIG_DIR}/{name}', 'figures')
    print(f'[fig] Saved {name}')

import torchaudio
from IPython.display import Audio as IPAudio, display

def play(audio, sr, label=''):
    if hasattr(audio, 'numpy'): audio = audio.squeeze().numpy()
    print(f'  {label}  ({len(audio)/sr:.1f}s | sr={sr})')
    display(IPAudio(audio, rate=int(sr)))

def save_audio(audio, sr, filename):
    path = f'{AUDIO_DIR}/{filename}'
    if not isinstance(audio, torch.Tensor): audio = torch.tensor(audio)
    torchaudio.save(path, audio.squeeze().unsqueeze(0).float().cpu(), sr)
    print(f'[audio] Saved {filename}')

print('Core utilities ready.')


In [ ]:
def _load_summaries_from_drive():
    ckpt = load_latest_checkpoint('all_summaries')
    if ckpt and 'summaries' in ckpt:
        return {s['label']: s for s in ckpt['summaries']}
    return {}

ALL_SUMMARIES: dict = _load_summaries_from_drive()
print(f'Loaded {len(ALL_SUMMARIES)} existing summaries: {list(ALL_SUMMARIES.keys())}')

def store_summary(s):
    label = s['label']
    ALL_SUMMARIES[label] = s.copy()
    save_checkpoint({'summaries': list(ALL_SUMMARIES.values())}, 'all_summaries', 0)
    print(f'[summary] Stored {label} ({len(ALL_SUMMARIES)} total)')

def get_summaries():
    return sorted(ALL_SUMMARIES.values(), key=lambda s: s['label'])

def plot_phase_comparison(summaries=None, save_name='phase_comparison.png'):
    data = summaries or get_summaries()
    if not data: 
        print('No summaries yet.'); 
        return
    
    # Sort by label to ensure consistent ordering
    data = sorted(data, key=lambda s: s['label'])
    labels = [s['label'] for s in data]
    
    print(f'Plotting {len(data)} phases: {labels}')
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Textless S2ST Compression Pipeline: Phase Comparison',
                 fontsize=15, fontweight='bold')
    metrics = [('avg_bleu', 'ASR-BLEU (higher=better)', '#2196F3'),
               ('avg_chrf', 'ASR-ChrF (higher=better)', '#4CAF50'),
               ('avg_rtf',  'RTF (lower=faster)',        '#FF9800'),
               ('params_M', 'Parameters (M)',            '#9C27B0')]
    
    for ax, (key, title, color) in zip(axes.flat, metrics):
        vals = [s.get(key, 0) for s in data]
        x_pos = range(len(labels))
        bars = ax.bar(x_pos, vals, color=color, alpha=0.85, edgecolor='white', width=0.7)
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Add value labels on bars
        for bar, v in zip(bars, vals):
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2, height, 
                       f'{v:.1f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()
def plot_size_vs_quality(summaries=None, save_name='size_vs_quality.png'):
    data = summaries or get_summaries()
    if not data: return
    fig, ax = plt.subplots(figsize=(10, 7))
    params = [s['params_M'] for s in data]
    chrf   = [s['avg_chrf'] for s in data]
    bleu   = [s['avg_bleu'] for s in data]
    ax.scatter(params, bleu, s=120, c='#2196F3', zorder=5, label='ASR-BLEU')
    ax.scatter(params, chrf, s=120, c='#4CAF50', marker='s', zorder=5, label='ASR-ChrF')
    for i, lbl in enumerate([s['label'] for s in data]):
        ax.annotate(lbl, (params[i], bleu[i]), fontsize=7, xytext=(5,5),
                    textcoords='offset points')
    ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Score')
    ax.set_title('Model Size vs Translation Quality', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()

print('Plotting helpers ready.')


In [ ]:
"""
QUICK INTEGRATION SNIPPET
Copy-paste this entire cell into seamless-final.ipynb after the existing summary functions
This is a condensed version for immediate use
"""

# ============================================================================
# ENHANCED TRACKING - Insert after ALL_SUMMARIES definition
# ============================================================================

def _load_detailed_summaries_from_drive():
    ckpt = load_latest_checkpoint('all_detailed_summaries')
    if ckpt and 'detailed_summaries' in ckpt:
        return {s['label']: s for s in ckpt['detailed_summaries']}
    return {}

ALL_DETAILED_SUMMARIES = _load_detailed_summaries_from_drive()
print(f'Loaded {len(ALL_DETAILED_SUMMARIES)} detailed summaries')

def store_detailed_summary(s):
    label = s['label']
    ALL_DETAILED_SUMMARIES[label] = s.copy()
    save_checkpoint({'detailed_summaries': list(ALL_DETAILED_SUMMARIES.values())}, 
                    'all_detailed_summaries', 0)
    print(f'[detailed] Stored {label}')

def compute_detailed_summary(results, label, params_M):
    from collections import defaultdict
    by_pair = defaultdict(list)
    for r in results:
        if not math.isnan(r.get('rtf', float('nan'))):
            by_pair[f"{r['src_lang']}→{r['tgt_lang']}"].append(r)
    
    pair_stats = {}
    for pair_key, pair_results in by_pair.items():
        pair_stats[pair_key] = {
            'n_samples': len(pair_results),
            'avg_bleu': float(np.mean([r['bleu'] for r in pair_results])),
            'avg_chrf': float(np.mean([r['chrf'] for r in pair_results])),
            'avg_rtf': float(np.mean([r['rtf'] for r in pair_results])),
            'std_chrf': float(np.std([r['chrf'] for r in pair_results])),
        }
    
    valid = [r for r in results if not math.isnan(r.get('rtf', float('nan')))]
    by_src = defaultdict(list)
    by_tgt = defaultdict(list)
    for r in valid:
        by_src[r['src_lang']].append(r)
        by_tgt[r['tgt_lang']].append(r)
    
    return {
        'label': label, 'params_M': params_M, 'n_total': len(valid),
        'avg_bleu': float(np.mean([r['bleu'] for r in valid])),
        'avg_chrf': float(np.mean([r['chrf'] for r in valid])),
        'avg_rtf': float(np.mean([r['rtf'] for r in valid])),
        'std_chrf': float(np.std([r['chrf'] for r in valid])),
        'pair_stats': pair_stats,
        'by_src_lang': {lang: {
            'n_samples': len(rs),
            'avg_chrf': float(np.mean([r['chrf'] for r in rs])),
            'avg_bleu': float(np.mean([r['bleu'] for r in rs])),
        } for lang, rs in by_src.items()},
        'by_tgt_lang': {lang: {
            'n_samples': len(rs),
            'avg_chrf': float(np.mean([r['chrf'] for r in rs])),
            'avg_bleu': float(np.mean([r['bleu'] for r in rs])),
        } for lang, rs in by_tgt.items()},
    }

def plot_detailed_phase_comparison(save_name='detailed_comparison.png'):
    summaries = sorted(ALL_DETAILED_SUMMARIES.values(), key=lambda s: s['label'])
    if not summaries: 
        print('No detailed summaries yet.')
        return
    
    print(f'Plotting detailed comparison for {len(summaries)} phases: {[s["label"] for s in summaries]}')
    
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('Detailed Phase Comparison: Per-Language Breakdown', fontsize=14, fontweight='bold')
    
    labels = [s['label'] for s in summaries]
    
    # Panel 1: Overall ChrF/BLEU
    ax1 = plt.subplot(3, 3, 1)
    chrfs = [s['avg_chrf'] for s in summaries]
    bleus = [s['avg_bleu'] for s in summaries]
    x = np.arange(len(labels))
    ax1.bar(x - 0.2, chrfs, 0.4, label='ChrF', color='#4CAF50', alpha=0.85)
    ax1.bar(x + 0.2, bleus, 0.4, label='BLEU', color='#2196F3', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
    ax1.set_title('Overall Quality', fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Panel 2: Per-pair ChrF comparison across ALL phases (vertical bars, grouped by language pair)
    ax2 = plt.subplot(3, 3, 2)
    
    # Collect all unique language pairs across all phases
    all_pairs = set()
    for s in summaries:
        if 'pair_stats' in s:
            all_pairs.update(s['pair_stats'].keys())
    all_pairs = sorted(all_pairs)
    
    if all_pairs:
        n_pairs = len(all_pairs)
        n_phases = len(summaries)
        bar_width = 0.8 / n_phases
        x_pos = np.arange(n_pairs)
        
        for phase_idx, s in enumerate(summaries):
            pair_stats = s.get('pair_stats', {})
            chrf_vals = [pair_stats.get(pair, {}).get('avg_chrf', 0) for pair in all_pairs]
            offset = (phase_idx - n_phases/2 + 0.5) * bar_width
            ax2.bar(x_pos + offset, chrf_vals, bar_width, 
                   label=s['label'], alpha=0.85)
        
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels(all_pairs, rotation=45, ha='right', fontsize=6)
        ax2.set_ylabel('ASR-ChrF')
        ax2.set_title('ChrF by Language Pair (All Phases)', fontweight='bold', fontsize=9)
        ax2.legend(fontsize=6, ncol=2)
        ax2.grid(alpha=0.3, axis='y')
    
    # Panel 3: BLEU by pair for all phases
    ax3 = plt.subplot(3, 3, 3)
    if all_pairs:
        for phase_idx, s in enumerate(summaries):
            pair_stats = s.get('pair_stats', {})
            bleu_vals = [pair_stats.get(pair, {}).get('avg_bleu', 0) for pair in all_pairs]
            offset = (phase_idx - n_phases/2 + 0.5) * bar_width
            ax3.bar(x_pos + offset, bleu_vals, bar_width, 
                   label=s['label'], alpha=0.85)
        
        ax3.set_xticks(x_pos)
        ax3.set_xticklabels(all_pairs, rotation=45, ha='right', fontsize=6)
        ax3.set_ylabel('ASR-BLEU')
        ax3.set_title('BLEU by Language Pair (All Phases)', fontweight='bold', fontsize=9)
        ax3.legend(fontsize=6, ncol=2)
        ax3.grid(alpha=0.3, axis='y')
    
    # Panel 4: Source language trends
    ax4 = plt.subplot(3, 3, 4)
    if summaries and 'by_src_lang' in summaries[0]:
        # Get all source languages
        all_src_langs = set()
        for s in summaries:
            if 'by_src_lang' in s:
                all_src_langs.update(s['by_src_lang'].keys())
        all_src_langs = sorted(all_src_langs)
        
        for src in all_src_langs:
            src_chrfs = []
            for s in summaries:
                if 'by_src_lang' in s and src in s['by_src_lang']:
                    src_chrfs.append(s['by_src_lang'][src]['avg_chrf'])
                else:
                    src_chrfs.append(None)
            valid_x = [i for i, v in enumerate(src_chrfs) if v is not None]
            valid_y = [v for v in src_chrfs if v is not None]
            if valid_y:
                ax4.plot(valid_x, valid_y, 'o-', label=src.upper(), lw=2, ms=5)
        ax4.set_xticks(range(len(labels)))
        ax4.set_xticklabels(labels, rotation=30, ha='right', fontsize=6)
        ax4.set_ylabel('ASR-ChrF')
        ax4.set_title('Source Language Trends', fontweight='bold', fontsize=9)
        ax4.legend(fontsize=6, ncol=2)
        ax4.grid(alpha=0.3)
    
    # Panel 5: Target language trends
    ax5 = plt.subplot(3, 3, 5)
    if summaries and 'by_tgt_lang' in summaries[0]:
        all_tgt_langs = set()
        for s in summaries:
            if 'by_tgt_lang' in s:
                all_tgt_langs.update(s['by_tgt_lang'].keys())
        all_tgt_langs = sorted(all_tgt_langs)
        
        for tgt in all_tgt_langs:
            tgt_chrfs = []
            for s in summaries:
                if 'by_tgt_lang' in s and tgt in s['by_tgt_lang']:
                    tgt_chrfs.append(s['by_tgt_lang'][tgt]['avg_chrf'])
                else:
                    tgt_chrfs.append(None)
            valid_x = [i for i, v in enumerate(tgt_chrfs) if v is not None]
            valid_y = [v for v in tgt_chrfs if v is not None]
            if valid_y:
                ax5.plot(valid_x, valid_y, 's-', label=tgt.upper(), lw=2, ms=5)
        ax5.set_xticks(range(len(labels)))
        ax5.set_xticklabels(labels, rotation=30, ha='right', fontsize=6)
        ax5.set_ylabel('ASR-ChrF')
        ax5.set_title('Target Language Trends', fontweight='bold', fontsize=9)
        ax5.legend(fontsize=6, ncol=2)
        ax5.grid(alpha=0.3)
    
    # Panel 6: Params vs Quality
    ax6 = plt.subplot(3, 3, 6)
    params = [s['params_M'] for s in summaries]
    ax6.scatter(params, chrfs, s=100, c='#4CAF50', marker='o', label='ChrF', zorder=5)
    ax6.scatter(params, bleus, s=100, c='#2196F3', marker='s', label='BLEU', zorder=5)
    for i, lbl in enumerate(labels):
        ax6.annotate(lbl, (params[i], chrfs[i]), fontsize=6, xytext=(3,3),
                    textcoords='offset points')
    ax6.set_xlabel('Parameters (M)')
    ax6.set_ylabel('Score')
    ax6.set_title('Size vs Quality', fontweight='bold', fontsize=9)
    ax6.legend(fontsize=7)
    ax6.grid(alpha=0.3)
    
    # Panel 7: Speaker sim by pair (if available)
    ax7 = plt.subplot(3, 3, 7)
    ax7.text(0.5, 0.5, 'Reserved for\nSpeaker Similarity', 
            ha='center', va='center', transform=ax7.transAxes, fontsize=10)
    ax7.axis('off')
    
    # Panel 8: RTF comparison
    ax8 = plt.subplot(3, 3, 8)
    rtfs = [s['avg_rtf'] for s in summaries]
    bars = ax8.bar(range(len(labels)), rtfs, color='#FF9800', alpha=0.85, edgecolor='white')
    ax8.set_xticks(range(len(labels)))
    ax8.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
    ax8.set_ylabel('RTF (lower=faster)')
    ax8.set_title('Inference Speed', fontweight='bold', fontsize=9)
    ax8.grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, rtfs):
        if v > 0:
            ax8.text(bar.get_x()+bar.get_width()/2, bar.get_height(), 
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6)
    
    # Panel 9: Summary table
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    table_data = [['Phase', 'Params(M)', 'ChrF', 'BLEU', 'RTF']]
    for s in summaries:
        table_data.append([
            s['label'][:12],
            f"{s['params_M']:.0f}",
            f"{s['avg_chrf']:.1f}",
            f"{s['avg_bleu']:.1f}",
            f"{s['avg_rtf']:.3f}"
        ])
    tbl = ax9.table(cellText=table_data[1:], colLabels=table_data[0],
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1.0, 1.5)
    ax9.set_title('Summary Table', fontweight='bold', fontsize=9, pad=10)
    
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()
    print(f'✓ Detailed comparison plotted for {len(summaries)} phases')
def print_detailed_summary_table(phase_label=None):
    summaries = sorted(ALL_DETAILED_SUMMARIES.values(), key=lambda s: s['label'])
    if not summaries: return
    summary = next((s for s in summaries if s['label'] == phase_label), summaries[-1]) if phase_label else summaries[-1]
    
    print(f'\n{"="*80}\n  {summary["label"]} - {summary["params_M"]:.1f}M params\n{"="*80}')
    print(f'Overall: ChrF={summary["avg_chrf"]:.2f}±{summary.get("std_chrf",0):.2f}  '
          f'BLEU={summary["avg_bleu"]:.2f}  RTF={summary["avg_rtf"]:.4f}')
    
    pair_stats = summary.get('pair_stats', {})
    if pair_stats:
        print(f'\nPer-Pair ({len(pair_stats)} pairs):')
        print(f'  {"Pair":<15} {"N":>4} {"ChrF":>8} {"BLEU":>8} {"RTF":>8}')
        for pair in sorted(pair_stats.keys()):
            s = pair_stats[pair]
            print(f'  {pair:<15} {s["n_samples"]:>4} {s["avg_chrf"]:>8.2f} '
                  f'{s["avg_bleu"]:>8.2f} {s["avg_rtf"]:>8.4f}')
    
    by_src = summary.get('by_src_lang', {})
    if by_src:
        print(f'\nBy Source Language:')
        for lang in sorted(by_src.keys()):
            s = by_src[lang]
            print(f'  {lang.upper():>6}: ChrF={s["avg_chrf"]:>6.2f}  BLEU={s["avg_bleu"]:>6.2f}  (n={s["n_samples"]})')
    
    by_tgt = summary.get('by_tgt_lang', {})
    if by_tgt:
        print(f'\nBy Target Language:')
        for lang in sorted(by_tgt.keys()):
            s = by_tgt[lang]
            print(f'  {lang.upper():>6}: ChrF={s["avg_chrf"]:>6.2f}  BLEU={s["avg_bleu"]:>6.2f}  (n={s["n_samples"]})')
    print('='*80)

print('✓ Enhanced tracking loaded: store_detailed_summary(), compute_detailed_summary(), plot_detailed_phase_comparison(), print_detailed_summary_table()')


## 📊 Enhanced Per-Language Tracking Enabled

**New functions available:**
- `compute_detailed_summary(results, label, params_M)` - Extract per-language metrics
- `store_detailed_summary(summary)` - Save to checkpoint
- `plot_detailed_phase_comparison()` - 9-panel visualization
- `print_detailed_summary_table(phase_label)` - Text output

**To use in benchmark cells:**
```python
# After running benchmark
p0_results, p0_summary = run_benchmark_asr(model, samples, 'P0_Label', save_n=4)
p0_detailed = compute_detailed_summary(p0_results, 'P0_Label', p0_summary['params_M'])

# Save both
save_checkpoint({
    'results': p0_results,
    'summary': p0_summary,
    'detailed_summary': p0_detailed  # NEW
}, 'phase0_benchmark', 0)

store_summary(p0_summary)
store_detailed_summary(p0_detailed)  # NEW
print_detailed_summary_table('P0_Label')  # NEW
plot_detailed_phase_comparison()  # NEW
```

**All per-language data now preserved in checkpoints!**

In [ ]:
# ── MMS-ASR for Bengali, Hindi, Arabic ──────────────────────────────────────
import gc as _stdlib_gc

_MMS_MODEL_ID = 'facebook/mms-1b-all'
_mms_asr_models = {}  # Cache models per language
_mms_asr_processors = {}

def _ensure_mms_loaded(lang_code):
    """Load MMS model for specific language (ben, hin, ara)"""
    global _mms_asr_models, _mms_asr_processors
    if lang_code in _mms_asr_models: return
    
    from transformers import Wav2Vec2ForCTC, AutoProcessor
    print(f'[MMS-ASR] Loading {_MMS_MODEL_ID} lang={lang_code}...')
    _mms_asr_processors[lang_code] = AutoProcessor.from_pretrained(
        _MMS_MODEL_ID, target_lang=lang_code)
    _mms_asr_models[lang_code] = Wav2Vec2ForCTC.from_pretrained(
        _MMS_MODEL_ID, target_lang=lang_code,
        ignore_mismatched_sizes=True, torch_dtype=torch.float16)
    _mms_asr_models[lang_code].load_adapter(lang_code)
    _mms_asr_models[lang_code] = _mms_asr_models[lang_code].eval()
    try: 
        _mms_asr_models[lang_code] = _mms_asr_models[lang_code].to('cuda:0')
    except RuntimeError: 
        pass
    print(f'[MMS-ASR] {lang_code} ready.')

def asr_transcribe_mms(audio_np, lang_code, sr=16000):
    _ensure_mms_loaded(lang_code)
    if audio_np is None or len(audio_np) < 400:
        return ''

    if sr != 16000:
        audio_np = torchaudio.functional.resample(
            torch.tensor(audio_np), sr, 16000).numpy()

    model = _mms_asr_models[lang_code]
    processor = _mms_asr_processors[lang_code]

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    inputs = processor(audio_np, sampling_rate=16000, return_tensors='pt')
    input_values = inputs.input_values.to(device).to(dtype)

    with torch.no_grad():
        logits = model(input_values=input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)[0].strip()

# ── Whisper-medium for English and Chinese ──────────────────────────────────
_whisper_model = None
_whisper_processor = None

def _ensure_whisper_loaded():
    global _whisper_model, _whisper_processor
    if _whisper_model is not None: return
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    print('[Whisper] Loading openai/whisper-medium...')
    _whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-medium')
    _whisper_model = WhisperForConditionalGeneration.from_pretrained(
        'openai/whisper-medium', torch_dtype=torch.float16)
    _whisper_model = _whisper_model.eval()
    try:
        device = 'cuda:1' if N_GPU > 1 else 'cuda:0'
        _whisper_model = _whisper_model.to(device)
    except RuntimeError:
        pass
    print('[Whisper] Ready.')

def asr_transcribe_whisper(audio_np, lang='en', sr=16000):
    """
    Transcribe audio using Whisper-medium for English or Chinese.
    lang: 'en' for English, 'zh' for Chinese
    """
    _ensure_whisper_loaded()
    if audio_np is None or len(audio_np) < 400: return ''
    
    # Resample if needed
    if sr != 16000:
        audio_np = torchaudio.functional.resample(
            torch.tensor(audio_np), sr, 16000).numpy()
    
    device = next(_whisper_model.parameters()).device
    dtype = next(_whisper_model.parameters()).dtype
    
    # Whisper language codes
    whisper_lang = 'en'
    
    try:
        # Process audio - ensure correct dtype
        inputs = _whisper_processor(
            audio_np, 
            sampling_rate=16000, 
            return_tensors='pt',
            return_attention_mask=True)
        
        # Move to device and convert to model dtype
        input_features = inputs['input_features'].to(device).to(dtype)
        
        # Use modern task/language parameters instead of forced_decoder_ids
        with torch.no_grad():
            predicted_ids = _whisper_model.generate(
                input_features,
                language=whisper_lang,
                task='transcribe',
                max_new_tokens=256,
                num_beams=1,
                do_sample=False)
        
        transcription = _whisper_processor.batch_decode(
            predicted_ids, skip_special_tokens=True)[0]
        return transcription.strip()
    except Exception as e:
        print(f'[Whisper] Error: {e}')
        import traceback
        traceback.print_exc()
        return ''

# ── M4T lang → ASR backend mapping ──────────────────────────────────────────
M4T_FLEURS_MAP = {
    'eng': 'en_us', 'ben': 'bn_in', 'cmn': 'cmn_hans_cn',
    'arb': 'ar_eg', 'hin': 'hi_in',
}

# MMS language codes - UPDATED to include Chinese
MMS_LANG_MAP = {
    'ben': 'ben',  # Bengali
    'hin': 'hin',  # Hindi
    'arb': 'ara',  # Arabic (MMS uses 'ara' for Arabic)
    'cmn': 'cmn',  # Chinese Mandarin (ADDED)
}

# UPDATED: Whisper only for English, MMS for all others
LANG_ASR_CONFIG = {
    'ben': ('mms', 'ben'),       # MMS for Bengali
    'hin': ('mms', 'hin'),       # MMS for Hindi
    'arb': ('mms', 'ara'),       # MMS for Arabic
    'cmn': ('mms', 'cmn-script_simplified'),       # MMS for Chinese (CHANGED from Whisper)
    'eng': ('whisper', 'en'),    # Whisper for English only
}

def asr_transcribe(audio_np, tgt_lang_m4t, sr=16000):
    """Route to correct ASR backend: Whisper for EN only, MMS for all others"""    
    if audio_np is None or len(audio_np) < 800: return ''
    backend, lang_code = LANG_ASR_CONFIG.get(tgt_lang_m4t)  # Default to MMS
    try:
        if backend == 'mms':
            return asr_transcribe_mms(audio_np, lang_code, sr)
        else:  # whisper (only for English now)
            return asr_transcribe_whisper(audio_np, lang_code, sr)
    except Exception as e:
        print(f'[ASR] Error ({tgt_lang_m4t}): {e}')
        return ''


print('ASR stack ready:')
print('  - Whisper-medium: English only')
print('  - MMS-1b-all: Bengali, Hindi, Arabic, Chinese')

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
_bleu = BLEU(effective_order=True)
_chrf = CHRF()

def find_layers_attr(component):
    for attr in ['layers', 'layer', 'inner_layers', 'encoder_layers', 'decoder_layers']:
        if hasattr(component, attr): return attr
    return None

def compute_bleu(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _bleu.sentence_score(hyp.strip(), [ref.strip()]).score

def compute_chrf(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _chrf.sentence_score(hyp.strip(), [ref.strip()]).score

def _remap_ids_for_decode(mdl, ids):
    if hasattr(mdl, '_vocab_remap_to_old'):
        remap = mdl._vocab_remap_to_old
        ids = ids.clone()
        mask = (ids >= 0) & (ids < len(remap))
        ids[mask] = remap[ids[mask]]
    return ids

def _model_input_device(mdl):
    if hasattr(mdl, 'speech_encoder'):
        return next(mdl.speech_encoder.parameters()).device
    return next(mdl.parameters()).device

def run_s2st(mdl, wav, tgt_lang='ben'):
    """Full S2ST for models with text decoder (Phases 0-3)."""
    inputs = processor(audio=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    with torch.no_grad():
        try:
            out  = mdl.generate(**inputs, tgt_lang=tgt_lang,
                                return_intermediate_token_ids=True)
            text_ids = _remap_ids_for_decode(mdl, out.sequences.cpu())
            text = processor.batch_decode(text_ids, skip_special_tokens=True)[0]
            wav_out = out.waveform.cpu().numpy().squeeze() if out.waveform is not None else np.zeros(16000)
            return text, wav_out
        except RuntimeError:
            return run_s2t_only(mdl, wav, tgt_lang), np.zeros(16000)

def run_s2t_only(mdl, wav, tgt_lang='ben'):
    """Text-only generation (for benchmarking text-decoder models)."""
    inputs = processor(audio=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    orig_voc = mdl.vocoder
    inp_dev  = next(iter(inputs.values())).device
    class _Noop(nn.Module):
        def forward(self, *a, **kw): return torch.zeros(1,1,device=inp_dev), [1]
    mdl.vocoder = _Noop()
    try:
        with torch.no_grad():
            out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                               return_intermediate_token_ids=True)
    finally:
        mdl.vocoder = orig_voc
    text_ids = _remap_ids_for_decode(mdl, out.sequences.cpu())
    return processor.batch_decode(text_ids, skip_special_tokens=True)[0]

def quick_eval_chrf(mdl, samples, max_samples=16, group_size=25):
    """
    Optimized: Only load audio for samples we actually use.
    """
    scores = []
    num_langs = len(samples) // group_size
    per_lang = max(1, max_samples // num_langs)
    
    for i in range(num_langs):
        start = i * group_size
        
        # ✅ OPTIMIZED: Only load the samples we need
        for j in range(per_lang):
            
            idx = start + j
            if idx >= len(samples):
                break
            
            s = samples[idx]  # Load only this one sample
            tgt = s.get('tgt_lang', 'ben')
            _, wav_out = run_s2st(mdl, s['wav'], tgt_lang=tgt)
            pred = asr_transcribe(wav_out, tgt)
            scores.append(compute_chrf(pred, s['ref']))
    
    return float(np.mean(scores))


print('Benchmark functions ready.')

In [ ]:
import jieba

def zh_tokenize(text):
    return " ".join(jieba.lcut(text.replace(" ", "")))


def run_benchmark_asr(mdl, samples, label='model', save_n=4):
    """ASR-based benchmark: translate audio → ASR transcribe → compute ASR-ChrF/BLEU."""
    print(f'\n{"="*60}\n  BENCHMARK (ASR): {label}  Samples:{len(samples)}\n{"="*60}')
    gpu_mem()
    results = []
    
    # Group samples by language pair for organized output
    from collections import defaultdict
    by_pair = defaultdict(list)
    for s in samples:
        by_pair[f"{s['src_lang']}→{s['tgt_lang']}"].append(s)
    
    for pair_key, pair_samples in by_pair.items():
        print(f'\n  === {pair_key} ({len(pair_samples)} samples) ===')
        for i, s in enumerate(pair_samples):
            try:
                dur = len(s['wav']) / 16000
                t0  = time.time()
                # Run S2ST translation
                _, wav_out = run_s2st(mdl, s['wav'], tgt_lang=s['tgt_lang'])
                rtf  = (time.time() - t0) / dur
                
                # ASR transcribe output audio
                pred = asr_transcribe(wav_out, s['tgt_lang'])
                
                ref = s['ref']
                hyp = pred
                if s['tgt_lang'] == 'cmn':
                    print("bench cmn")
                    ref_clean = ref.replace(" ", "")
                    hyp_clean = hyp.replace(" ", "")
                
                    # BLEU (tokenized)
                    ref_bleu = zh_tokenize(ref_clean)
                    hyp_bleu = zh_tokenize(hyp_clean)
                
                    # chrF (raw)
                    ref_chrf = ref_clean
                    hyp_chrf = hyp_clean

                    bleu = compute_bleu(hyp_bleu, ref_bleu)
                    chrf = compute_chrf(hyp_chrf, ref_chrf)    
                else:
                    bleu = compute_bleu(pred, ref)
                    chrf = compute_chrf(pred, ref)

                print(f'  [{i+1:>2}/{len(pair_samples)}] ASR-BLEU={bleu:5.1f} ASR-ChrF={chrf:5.1f} RTF={rtf:.3f}')
                print(f'              pred: {pred[:80]}')
                
                if save_n > 0 and i < save_n:
                    play(s['wav'], 16000, label=f'{label}_{pair_key}_s{i+1}in.wav')
                    save_audio(s['wav'], 16000, f'{label}_{pair_key}_s{i+1}in.wav')
                    play(wav_out, 16000, label=f'{label}_{pair_key}_s{i+1}out.wav')
                    save_audio(wav_out, 16000, f'{label}_{pair_key}_s{i+1}out.wav')
                
                results.append(dict(
                    id=s['id'], src_lang=s['src_lang'], tgt_lang=s['tgt_lang'],
                    bleu=bleu, chrf=chrf, rtf=rtf, pred=pred, ref=s['ref']))
            except Exception as e:
                import traceback; traceback.print_exc()
                results.append(dict(
                    id=s['id'], src_lang=s.get('src_lang','?'), tgt_lang=s.get('tgt_lang','?'),
                    bleu=0, chrf=0, rtf=float('nan'), pred='', ref=s.get('ref','')))
    
    valid = [r for r in results if not math.isnan(r['rtf'])]
    summary = dict(
        label=label, n=len(valid),
        avg_bleu=float(np.mean([r['bleu'] for r in valid])) if valid else 0,
        avg_chrf=float(np.mean([r['chrf'] for r in valid])) if valid else 0,
        avg_rtf =float(np.mean([r['rtf']  for r in valid])) if valid else 0,
        params_M=count_params(mdl)
    )
    
    # Per-pair breakdown
    print(f'\n  === Summary by Language Pair ===')
    for pair_key in by_pair.keys():
        pair_res = [r for r in valid if f"{r['src_lang']}→{r['tgt_lang']}" == pair_key]
        if pair_res:
            avg_chrf_pair = np.mean([r['chrf'] for r in pair_res])
            avg_bleu_pair = np.mean([r['bleu'] for r in pair_res])
            print(f'  {pair_key:<12} ASR-ChrF={avg_chrf_pair:5.2f}  ASR-BLEU={avg_bleu_pair:5.2f}')
    
    print(f'\n  Overall: ASR-BLEU={summary["avg_bleu"]:.2f} ASR-ChrF={summary["avg_chrf"]:.2f}'
          f' RTF={summary["avg_rtf"]:.4f} Params={summary["params_M"]:.1f}M')
    return results, summary

# Alias for backward compatibility
run_benchmark = run_benchmark_asr


In [ ]:
from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor

try:
    HF_TOKEN = _get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(HF_TOKEN)
    print('Logged into HuggingFace Hub.')
except Exception as e:
    print(f'HF login skipped: {e}')

MODEL_NAME = 'facebook/seamless-m4t-v2-large'
processor = None   # Will be set when loading any model

def load_base_model():
    global processor
    print(f'Loading processor from {MODEL_NAME}...')
    proc = SeamlessM4TProcessor.from_pretrained(MODEL_NAME)
    print(f'Loading model -- may take 5-10 min...')
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
    mdl.eval()
    print('Model loaded.'); gpu_mem()
    processor = proc
    return mdl, proc

print('load_base_model() ready. Call it to load teacher/base model.')


In [ ]:
## Dataset loading — battle-tested from seamless-cse465v5 (Cells 24-26)
import concurrent.futures, io, soundfile as sfile, pandas as pd

# LOCAL_PARQUET_CACHE = '/kaggle/working/fleurs_parquet'
LOCAL_PARQUET_CACHE = '/kaggle/input/datasets/rayedriasat/fleurs5'
BASE_PARQUET_URL = 'https://huggingface.co/datasets/google/fleurs/resolve/refs%2Fconvert%2Fparquet'
DRIVE_FLEURS_PATH = f'{GDRIVE_ROOT}/fleurs_parquet'

def _list_parquet_urls(lang, split):
    import requests
    urls, i = [], 0
    while True:
        url = f'{BASE_PARQUET_URL}/{lang}/{split}/{i:04d}.parquet?download=true'
        try:
            r = requests.head(url, timeout=15, allow_redirects=True)
            if r.status_code == 200: urls.append(url); i += 1
            else: break
        except: break
    if not urls:
        urls = [f'{BASE_PARQUET_URL}/{lang}/{split}/0000.parquet?download=true']
        print(f'  [WARN] fallback to shard 0000 for {lang}/{split}')
    print(f'  [shards] {lang}/{split}: {len(urls)} shard(s)')
    return urls

def _download_shard(args):
    import requests
    url, dest = args
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 1024*1024:
        return url, True, 'cached'
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(8*1024*1024):
                    if chunk: f.write(chunk)
            if dest.stat().st_size > 1024*1024: return url, True, 'downloaded'
            raise RuntimeError('Downloaded file too small')
        except Exception as e:
            if dest.exists(): dest.unlink()
            if attempt == 2: return url, False, str(e)
    return url, False, 'unknown'

def load_fleurs_parallel(src_lang, tgt_lang, split='train', n_workers=4):
    from datasets import Dataset
    tasks = []
    for lang in [src_lang, tgt_lang]:
        urls = _list_parquet_urls(lang, split)
        for i, url in enumerate(urls):
            dest = f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_{i:04d}.parquet'
            tasks.append((url, dest))
    print(f'[Parallel] Downloading {len(tasks)} shards...')
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        for url, ok, msg in pool.map(_download_shard, tasks):
            print(f'  {"OK" if ok else "FAIL"}: {msg}')
    def _load_lang(lang):
        if lang == "": 
            return None
        files = sorted(glob.glob(f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_*.parquet'))
        if not files: raise FileNotFoundError(f'No cached shards for {lang}')
        return Dataset.from_pandas(pd.read_parquet(files[0]))
    return _load_lang(src_lang), _load_lang(tgt_lang)

def push_fleurs_to_drive():
    if not ON_KAGGLE: return
    subprocess.run(f'rclone copy "{LOCAL_PARQUET_CACHE}/" "{DRIVE_FLEURS_PATH}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                   shell=True, capture_output=True)

def load_fleurs_from_drive(src_lang, tgt_lang, split='train'):
    from datasets import Dataset
    if not ON_KAGGLE: return None, None
    if not os.path.exists(LOCAL_PARQUET_CACHE):
        r = subprocess.run(f'rclone copy "{DRIVE_FLEURS_PATH}/" "{LOCAL_PARQUET_CACHE}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0: return None, None
    def _load_lang(lang):
        if lang == "": 
            return None
        files = sorted(glob.glob(f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_*.parquet'))
        if not files: return None
        return Dataset.from_pandas(pd.concat([pd.read_parquet(f) for f in files], ignore_index=True))
    src_ds = _load_lang(src_lang); tgt_ds = _load_lang(tgt_lang)
    if src_ds and tgt_ds: print(f'[gdrive] Loaded: {len(src_ds)} src, {len(tgt_ds)} tgt')
    return src_ds, tgt_ds

def _load_wav(audio_cell):
    """Verbatim from v5 Cell 25 — handles both HF Dataset and parquet byte formats."""    
    audio = audio_cell
    if isinstance(audio, dict) and 'array' in audio:
        arr, sr = audio['array'], audio['sampling_rate']
    elif isinstance(audio, dict) and 'bytes' in audio:
        wav, sr = sfile.read(io.BytesIO(audio['bytes']))
        if wav.ndim > 1: wav = wav.mean(axis=1)
        arr = wav
    else:
        raise RuntimeError(f'Unsupported audio format: {type(audio)}')
    arr = np.array(arr, dtype=np.float32)
    if sr != 16000:
        arr = torchaudio.functional.resample(torch.tensor(arr), sr, 16000).numpy()
    return arr

print('FLEURS data loaders ready.')


In [ ]:
# from datasets import Dataset
# tasks = []
# split = 'validation'
# for lang in ['ar_eg', 'bn_in', 'cmn_hans_cn', 'en_us', 'hi_in']:
#     urls = _list_parquet_urls(lang, split)
#     for i, url in enumerate(urls):
#         dest = f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_{i:04d}.parquet'
#         tasks.append((url, dest))
# print(f'[Parallel] Downloading {len(tasks)} shards...')
# with concurrent.futures.ThreadPoolExecutor(max_workers=16) as pool:
#     for url, ok, msg in pool.map(_download_shard, tasks):
#         print(f'  {"OK" if ok else "FAIL"}: {msg}')
# push_fleurs_to_drive()

In [ ]:
import os, glob, torch
from datetime import datetime

def session_status():
    print('=' * 65)
    print(f'  Platform : {PLATFORM}   Time : {datetime.now():%Y-%m-%d %H:%M}')
    if os.path.exists(CKPT_DIR):
        files = [f for f in glob.glob(f'{CKPT_DIR}/**/*.pt', recursive=True) if os.path.isfile(f)]
        print(f'  Checkpoint files: {len(files)}')
        for f in sorted(files)[:20]:
            print(f'    {os.path.relpath(f,CKPT_DIR):<50} {os.path.getsize(f)/1e6:>8.1f} MB')
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f'  GPU: {torch.cuda.get_device_name(0)}  VRAM: {props.total_memory/1e9:.1f} GB')
    print('=' * 65)

if not os.path.exists(LOCAL_PARQUET_CACHE):
    r = subprocess.run(f'rclone copy "{DRIVE_FLEURS_PATH}/" "{LOCAL_PARQUET_CACHE}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                       shell=True, capture_output=True, text=True)

sync_checkpoints_from_drive()
session_status()
print('\n✓ ALL SETUP CELLS COMPLETE — proceed to phases.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RAM-Efficient Parquet Streaming Dataset
# Loads audio on-demand, not during initialization
# RAM: ~4MB for 4000 samples (vs ~20GB with old approach)
# ══════════════════════════════════════════════════════════════════════════════

import pyarrow.parquet as pq

class ParquetStreamingDataset:
    """Memory-efficient dataset that streams from parquet files."""
    
    def __init__(self, parquet_cache_dir, src_lang, tgt_lang, split='train', 
                 max_samples_per_pair=500):
        self.cache_dir = pathlib.Path(parquet_cache_dir)
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.split = split
        self.max_samples = max_samples_per_pair
        self.samples = []
        self._build_index()
    
    def _build_index(self):
        """Build lightweight index (metadata only, no audio)."""
        src_files = sorted(self.cache_dir.glob(f'{M4T_FLEURS_MAP.get(self.src_lang)}/{self.split}_*.parquet'))
        tgt_files = sorted(self.cache_dir.glob(f'{M4T_FLEURS_MAP.get(self.tgt_lang)}/{self.split}_*.parquet'))
        
        if not src_files or not tgt_files:
            print(f'  WARNING: No parquet files for {self.src_lang}/{self.tgt_lang}')
            return
        
        # Read only ID columns (fast, <1MB RAM)
        src_ids = []
        for f in src_files:
            df = pd.read_parquet(f, columns=['id'])
            src_ids.extend([(str(f), idx, row_id) for idx, row_id in enumerate(df['id'])])
        
        tgt_ids = []
        for f in tgt_files:
            df = pd.read_parquet(f, columns=['id', 'transcription'])
            df = df[df['transcription'].str.strip().str.len() > 0]
            tgt_ids.extend([(str(f), idx, row_id, trans) 
                           for idx, (row_id, trans) in enumerate(zip(df['id'], df['transcription']))])
        
        # Create lookup dicts
        src_lookup = {row_id: (f, idx) for f, idx, row_id in src_ids}
        tgt_lookup = {row_id: (f, idx, trans) for f, idx, row_id, trans in tgt_ids}
        
        # Find matching IDs
        common_ids = set(src_lookup.keys()) & set(tgt_lookup.keys())
        
        # Build sample index (metadata only)
        for sample_id in list(common_ids)[:self.max_samples]:
            src_file, src_idx = src_lookup[sample_id]
            tgt_file, tgt_idx, tgt_text = tgt_lookup[sample_id]
            
            self.samples.append({
                'id': f"{self.src_lang}2{self.tgt_lang}_{sample_id}",
                'src_lang': self.src_lang,
                'tgt_lang': self.tgt_lang,
                'ref': tgt_text,
                '_src_file': src_file,
                '_src_idx': src_idx,
            })
        
        print(f'  Indexed {len(self.samples)} samples from {self.src_lang}→{self.tgt_lang}')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        """Get sample with audio loaded on-demand."""
        sample = self.samples[idx].copy()
        
        # Load audio only when accessed
        if '_src_file' in sample:
            audio = self._load_audio_from_parquet(
                sample['_src_file'], 
                sample['_src_idx']
            )
            sample['wav'] = audio
            del sample['_src_file']
            del sample['_src_idx']
        
        return sample
    
    def _load_audio_from_parquet(self, parquet_file, row_idx):
        """Load single audio sample from parquet file."""
        table = pq.read_table(parquet_file, columns=['audio'])
        audio_cell = table.to_pandas().iloc[row_idx]['audio']
        return _load_wav(audio_cell)


class MultilingualStreamingDataset:
    """Combines multiple language pairs into a single streaming dataset."""
    
    def __init__(self, parquet_cache_dir, lang_pairs, split='train', 
                 max_samples_per_pair=25):
        self.datasets = []
        
        for src_lang, tgt_lang in lang_pairs:
            ds = ParquetStreamingDataset(
                parquet_cache_dir, src_lang, tgt_lang, split, max_samples_per_pair
            )
            if len(ds) > 0:
                self.datasets.append(ds)
        
        # Build flat index
        self.index = []
        for ds_idx, ds in enumerate(self.datasets):
            for sample_idx in range(len(ds)):
                self.index.append((ds_idx, sample_idx))
        
        print(f'\n✓ Multilingual dataset ready: {len(self.index)} total samples')
        print(f'  RAM usage: ~{len(self.index) * 0.001:.1f} MB (metadata only)')
    
    def __len__(self):
        return len(self.index)
    
    def __getitem__(self, idx):
        """Get sample from appropriate sub-dataset."""
        if isinstance(idx, slice):
            indices = range(*idx.indices(len(self)))
            return [self[i] for i in indices]
        ds_idx, sample_idx = self.index[idx]
        return self.datasets[ds_idx][sample_idx]
    
    def __iter__(self):
        """Allow iteration."""
        for i in range(len(self)):
            yield self[i]

print('✓ Streaming dataset classes ready.')

In [ ]:
# ── Load Multilingual Eval Samples: En→X and X→En (all 5 languages) ─────────
# PLAN.md Section 5: 5 languages — EN, BN, ZH, AR, HI
N_EVAL_PER_PAIR = 25
EVAL_LANG_PAIRS = [
    ('eng', 'ben'), ('ben', 'eng'),  # English ↔ Bengali
    ('eng', 'cmn'), ('cmn', 'eng'),  # English ↔ Mandarin
    ('eng', 'arb'), ('arb', 'eng'),  # English ↔ Arabic
    ('eng', 'hin'), ('hin', 'eng'),  # English ↔ Hindi
]

In [ ]:
# ── Load Multilingual Eval Samples: En→X and X→En (all 5 languages) ──────────
# STREAMING VERSION: Only loads audio when accessed
# RAM: ~200KB for 200 samples (vs ~1GB with old approach)

print('Loading evaluation samples (streaming mode)...')
eval_samples = MultilingualStreamingDataset(
    parquet_cache_dir=LOCAL_PARQUET_CACHE,
    lang_pairs=EVAL_LANG_PAIRS,
    split='test',
    max_samples_per_pair=N_EVAL_PER_PAIR
)

print(f'\n✓ Loaded {len(eval_samples)} multilingual eval samples')
print(f'  Language pairs: {len(EVAL_LANG_PAIRS)}')
print(f'  RAM usage: ~{len(eval_samples) * 0.001:.1f} MB (metadata only)')

# Test: Load one sample to verify it works
test_sample = eval_samples[26]
print(f'\n✓ Test sample loaded:')
print(f'  ID: {test_sample["id"]}')
print(f'  Audio shape: {test_sample["wav"].shape}')
print(f'  Reference: {test_sample["ref"][:50]}...')

play(test_sample["wav"], 16000, label='hello.wav')

In [ ]:
# ── Load Multilingual Training Samples: En→X and X→En (all 5 languages) ─────
N_TRAIN_PER_PAIR = 1200  # 500 samples per direction = 4000 total

In [ ]:
# ── Load Multilingual Training Samples: En→X and X→En (all 5 languages) ──────
# STREAMING VERSION: Only loads audio when accessed
# RAM: ~4MB for 4000 samples (vs ~20GB with old approach)

print('Loading training samples (streaming mode)...')
ft_samples = MultilingualStreamingDataset(
    parquet_cache_dir=LOCAL_PARQUET_CACHE,
    lang_pairs=EVAL_LANG_PAIRS,
    split='train',
    max_samples_per_pair=N_TRAIN_PER_PAIR
)

print(f'\n✓ Loaded {len(ft_samples)} multilingual training samples')
print(f'  Language pairs: {len(EVAL_LANG_PAIRS)}')
print(f'  RAM usage: ~{len(ft_samples) * 0.001:.1f} MB (metadata only)')
print(f'  RAM saved: ~{len(ft_samples) * 5:.0f} MB (would be with old approach)')

# Summary by language pair
print('\nSamples per language pair:')
pair_counts = {}
for i in range(len(ft_samples)):
    sample_meta = ft_samples.datasets[ft_samples.index[i][0]].samples[ft_samples.index[i][1]]
    pair = f"{sample_meta['src_lang']}→{sample_meta['tgt_lang']}"
    pair_counts[pair] = pair_counts.get(pair, 0) + 1

for pair, count in sorted(pair_counts.items()):
    print(f'  {pair}: {count}')

In [ ]:
# ── Multilingual eval samples now integrated into eval_samples ──────────────
# All 5 languages (EN, BN, HI, ZH, AR) with bidirectional pairs are loaded above
print(f'Multilingual eval ready: {len(eval_samples)} samples across {len(EVAL_LANG_PAIRS)} pairs')
print(f'Language pairs: {EVAL_LANG_PAIRS}')

print(f'\n✓ Loaded {len(ft_samples)} multilingual training samples across {len(EVAL_LANG_PAIRS)} pairs')


# PHASE 6: Comprehensive Recovery with Multi-Stage LoRA + Sequence-Level KD
 Papers: LoRA (Hu ICLR 2022), rsLoRA (Kalajdzievski 2023), 
         Seq-KD (Kim & Rush EMNLP 2016), Moslem IWSLT 2025

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, TaskType
import math
import random
import numpy as np
from collections import defaultdict

# ── Load Models ───────────────────────────────────────────────────────────────
print('Loading Phase 5 student model (pruned)...')
model_student, processor = load_model_from_drive('phase5_dec_14L')
model_student = _consolidate_to_single_gpu(model_student)
device = torch.device('cuda:0')

print('Loading teacher model (V1 baseline for KD)...')
try:
    model_teacher, _ = load_model_from_drive('phase0_v1_baseline')
except:
    print('  Teacher not found, loading from HF...')
    model_teacher, _ = load_base_model()
model_teacher = _consolidate_to_single_gpu(model_teacher)
model_teacher.eval()
for p in model_teacher.parameters():
    p.requires_grad_(False)

print_model_breakdown(model_student, 'Student (Phase 5)')
print_model_breakdown(model_teacher, 'Teacher (V1)')

# ── LoRA Configuration (All Components) ───────────────────────────────────────
# Start with r=32 for T4 memory safety, can increase to r=64 if stable

lora_config_encoder = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        'linear_q', 'linear_k', 'linear_v', 'linear_out',  # Attention
        'intermediate_dense', 'output_dense',               # FFN
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.FEATURE_EXTRACTION,
    use_rslora=True,  # Rank-Stabilized LoRA
)

lora_config_decoder = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'out_proj',  # Self-attention
        'encoder_attn.q_proj', 'encoder_attn.k_proj',  # Cross-attention
        'encoder_attn.v_proj', 'encoder_attn.out_proj',
        'fc1', 'fc2',  # FFN
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    use_rslora=True,
)

lora_config_t2u = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'out_proj',
        'fc1', 'fc2',
    ],
    lora_dropout=0.05,
    bias='none',
    use_rslora=True,
)

# ── Freeze Base Model ─────────────────────────────────────────────────────────
for p in model_student.parameters():
    p.requires_grad_(False)

print('\n' + '='*80)
print('  PHASE 6: Multi-Stage Recovery Training')
print('='*80)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE A: T2U-Only Recovery (1 epoch, fast unit prediction restoration)
# ══════════════════════════════════════════════════════════════════════════════

print('\n[STAGE A] T2U-Only Recovery (restoring unit prediction quality)')
print('  Target: Fix "cracked voice" issue by improving unit sequences')

# Apply LoRA to T2U encoder + decoder
model_student.t2u_model.model.encoder = get_peft_model(
    model_student.t2u_model.model.encoder, lora_config_t2u)
model_student.t2u_model.model.decoder = get_peft_model(
    model_student.t2u_model.model.decoder, lora_config_t2u)

model_student.t2u_model.model.encoder.print_trainable_parameters()
model_student.t2u_model.model.decoder.print_trainable_parameters()

# Stage A hyperparameters
STAGE_A_STEPS = 500
BATCH_ACCUM_A = 8
LR_A = 2e-4

optimizer_a = torch.optim.AdamW([
    {'params': [p for p in model_student.t2u_model.parameters() if p.requires_grad],
     'lr': LR_A, 'weight_decay': 0.01}
], betas=(0.9, 0.98))

warmup_steps_a = int(0.1 * STAGE_A_STEPS)
def get_lr_scale_a(step):
    if step < warmup_steps_a:
        return step / warmup_steps_a
    return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps_a) / (STAGE_A_STEPS - warmup_steps_a)))

scheduler_a = torch.optim.lr_scheduler.LambdaLR(optimizer_a, get_lr_scale_a)
scaler_a = torch.cuda.amp.GradScaler()

model_student.train()
model_teacher.eval()
loss_log_a = []

print(f'\nTraining Stage A: {STAGE_A_STEPS} steps, LR={LR_A:.2e}')

for step in range(STAGE_A_STEPS):
    sample = ft_samples[random.randint(0, len(ft_samples)-1)]
    
    try:
        inputs = processor(audio=sample['wav'], sampling_rate=16000, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get teacher's unit sequence (pseudo-label for KD)
        with torch.no_grad():
            teacher_out = model_teacher.generate(
                **inputs, tgt_lang=sample['tgt_lang'],
                return_intermediate_token_ids=True)
            teacher_units = teacher_out.unit_sequences  # Target for T2U
        
        # Student forward pass (T2U only)
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            # Get speech encoder output (frozen)
            with torch.no_grad():
                enc_out = model_student.speech_encoder(**inputs).last_hidden_state
            
            # ── SeamlessM4T-v2 T2U architecture note ──────────────────────────
            # model_student.t2u_model is SeamlessM4Tv2TextToUnitForConditionalGeneration.
            # In v2 the T2U sub-model is NON-autoregressive and requires:
            #   • char_input_ids  – character-level tokens of the translated text
            #     (NOT inputs_embeds from the speech encoder).
            #   • The speech-encoder hidden states are NOT a valid input to t2u_model
            #     directly; t2u_model.model.encoder expects TEXT embeddings.
            # Calling t2u_model(inputs_embeds=speech_enc_out, ...) silently passes
            # through the encoder but the decoder's duration predictor and upsampler
            # receive wrong-shaped inputs → logits are never computed → loss=None.
            #
            # FIX: run the full student forward pass via model_student.forward()
            # (which internally calls speech_encoder → text_decoder → t2u_model in
            # the correct order), providing the teacher unit tokens as 'unit_labels'.
            # Alternatively, get the translated text tokens from the teacher and
            # tokenise them to char_input_ids for t2u_model directly (see branch B).
            #
            # We use approach A (full model forward with labels) here because it is
            # simpler and still isolates gradients to t2u LoRA adapters only.
            # ──────────────────────────────────────────────────────────────────

            if teacher_units is not None:
                # Approach A: full S2ST forward; only t2u LoRA params have grad.
                # labels=teacher_text_ids drives the text-decoder cross-entropy,
                # but since text_decoder has no requires_grad here Stage A won't
                # update it. The t2u branch computes loss against teacher_units.
                # Reuse teacher_out.sequences — the first generate() call above already
                # returns text token ids alongside unit_sequences when
                # return_intermediate_token_ids=True is set.
                # NOTE: generate_speech=False is only valid on SeamlessM4Tv2Model (the
                # combined top-level model), NOT on SeamlessM4Tv2ForSpeechToSpeech.
                # Passing it there raises: "model_kwargs not used: ['generate_speech']"
                teacher_text_ids = teacher_out.sequences  # (1, seq_len) — already obtained above

                # Prepare unit labels: strip BOS (index 0) to get targets
                unit_labels = teacher_units.to(device)  # (1, unit_seq_len)

                # Run student forward: speech_encoder + text_decoder + t2u_model
                # Pass unit_labels so t2u_model computes cross-entropy loss.
                full_out = model_student(
                    **inputs,
                    tgt_lang=sample['tgt_lang'],
                    labels=teacher_text_ids.to(device),   # text-decoder labels (frozen)
                )
                # full_out.loss is the TEXT decoder loss; we need the T2U loss.
                # Recompute T2U loss explicitly using logits from a second t2u pass
                # now that we have the correct char_input_ids from the model internals.
                # ── Approach B (direct t2u call with proper inputs) ────────────
                # Get the char-level token ids by processing teacher text through
                # the processor's char tokeniser, then call t2u_model properly.
                with torch.no_grad():
                    teacher_text_str = processor.tokenizer.batch_decode(
                        teacher_text_ids, skip_special_tokens=True)
                    char_inputs = processor.tokenizer(
                        teacher_text_str,
                        text_pair=None,
                        return_tensors='pt',
                        padding=True,
                        return_char_input_ids=True,   # SeamlessM4TProcessor kwarg
                    )
                    char_input_ids = char_inputs.get(
                        'char_input_ids', None)

                if char_input_ids is not None:
                    # Direct t2u_model call: char_input_ids → unit logits → CE loss
                    t2u_out = model_student.t2u_model(
                        input_ids=char_input_ids.to(device),
                        labels=unit_labels,
                    )
                    loss = t2u_out.loss if (hasattr(t2u_out, 'loss') and t2u_out.loss is not None) else torch.tensor(0.0, device=device, requires_grad=True)
                else:
                    # Fallback: distil from full-model logits via CE on unit head
                    # Get student T2U logits by running t2u_model with encoder_outputs
                    # obtained from the text decoder output
                    loss = full_out.loss if (hasattr(full_out, 'loss') and full_out.loss is not None) else torch.tensor(0.0, device=device, requires_grad=True)
            else:
                loss = torch.tensor(0.0, device=device, requires_grad=True)
        
        scaler_a.scale(loss / BATCH_ACCUM_A).backward()
        loss_log_a.append(loss.item())
        
        if (step + 1) % BATCH_ACCUM_A == 0:
            scaler_a.unscale_(optimizer_a)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.parameters() if p.requires_grad], 1.0)
            scaler_a.step(optimizer_a)
            scaler_a.update()
            optimizer_a.zero_grad()
            scheduler_a.step()
            torch.cuda.empty_cache()
        
        if (step + 1) % 50 == 0:
            recent_loss = np.mean(loss_log_a[-50:])
            cur_lr = optimizer_a.param_groups[0]['lr']
            print(f'  [A] Step {step+1:>4}/{STAGE_A_STEPS} | '
                  f'T2U_loss={recent_loss:.4f} | lr={cur_lr:.2e}')
        
        if (step + 1) % 250 == 0:
            save_checkpoint({
                'stage': 'A', 'step': step + 1,
                't2u_enc_state': model_student.t2u_model.model.encoder.state_dict(),
                't2u_dec_state': model_student.t2u_model.model.decoder.state_dict(),
                'optimizer_state': optimizer_a.state_dict(),
                'loss_log': loss_log_a,
            }, 'phase6_stage_a', step + 1)
    
    except Exception as e:
        print(f'  [A] Step {step+1} error: {e}')
        continue

print(f'\n✓ Stage A complete. Avg T2U loss: {np.mean(loss_log_a[-100:]):.4f}')



In [ ]:
# Quick eval after Stage A
print('\nStage A checkpoint evaluation...')
stage_a_chrf = quick_eval_chrf(model_student, eval_samples, max_samples=16)
print(f'  Stage A ChrF: {stage_a_chrf:.2f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE B: Text Decoder + T2U Joint Training (2-3 epochs, main recovery)
# ══════════════════════════════════════════════════════════════════════════════

print('\n[STAGE B] Text Decoder + T2U Joint Training (main quality recovery)')
print('  Mixed data: 50% teacher KD + 50% supervised')

# Apply LoRA to text decoder
model_student.text_decoder = get_peft_model(
    model_student.text_decoder, lora_config_decoder)
model_student.text_decoder.print_trainable_parameters()

# Stage B hyperparameters
STAGE_B_STEPS = 1500
BATCH_ACCUM_B = 8
LR_B = 1e-4

optimizer_b = torch.optim.AdamW([
    {'params': [p for p in model_student.text_decoder.parameters() if p.requires_grad],
     'lr': LR_B, 'weight_decay': 0.01},
    {'params': [p for p in model_student.t2u_model.parameters() if p.requires_grad],
     'lr': LR_B, 'weight_decay': 0.01},
], betas=(0.9, 0.98))

warmup_steps_b = int(0.1 * STAGE_B_STEPS)
def get_lr_scale_b(step):
    if step < warmup_steps_b:
        return step / warmup_steps_b
    return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps_b) / (STAGE_B_STEPS - warmup_steps_b)))

scheduler_b = torch.optim.lr_scheduler.LambdaLR(optimizer_b, get_lr_scale_b)
scaler_b = torch.cuda.amp.GradScaler()

loss_log_b = []
print(f'\nTraining Stage B: {STAGE_B_STEPS} steps, LR={LR_B:.2e}')

for step in range(STAGE_B_STEPS):
    sample = ft_samples[random.randint(0, len(ft_samples)-1)]
    use_kd = random.random() < 0.5  # 50% KD, 50% supervised
    
    try:
        inputs = processor(audio=sample['wav'], sampling_rate=16000, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        if use_kd:
            # Sequence-level KD: use teacher's text output as pseudo-label
            with torch.no_grad():
                teacher_out = model_teacher.generate(
                    **inputs, tgt_lang=sample['tgt_lang'],
                    return_intermediate_token_ids=True)
                teacher_text_ids = teacher_out.sequences
            
            labels = teacher_text_ids.to(device)
        else:
            # Supervised: use ground truth reference
            text_inputs = processor.tokenizer(
                sample['ref'], return_tensors='pt', padding=True, truncation=True)
            labels = text_inputs['input_ids'].to(device)
        
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model_student(
                **inputs, tgt_lang=sample['tgt_lang'], labels=labels)
            
            # Combined loss: S2TT (text decoder) + S2ST (T2U)
            s2tt_loss = outputs.loss if hasattr(outputs, 'loss') else torch.tensor(0.0).to(device)
            
            # T2U loss (if available)
            s2st_loss = torch.tensor(0.0).to(device)
            if hasattr(outputs, 't2u_loss'):
                s2st_loss = outputs.t2u_loss
            
            # Weighted combination (emphasize text quality for now)
            loss = 0.70 * s2tt_loss + 0.30 * s2st_loss
        
        scaler_b.scale(loss / BATCH_ACCUM_B).backward()
        loss_log_b.append({
            'total': loss.item(),
            's2tt': s2tt_loss.item(),
            's2st': s2st_loss.item(),
            'kd': use_kd
        })
        
        if (step + 1) % BATCH_ACCUM_B == 0:
            scaler_b.unscale_(optimizer_b)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.parameters() if p.requires_grad], 1.0)
            scaler_b.step(optimizer_b)
            scaler_b.update()
            optimizer_b.zero_grad()
            scheduler_b.step()
            torch.cuda.empty_cache()
        
        if (step + 1) % 50 == 0:
            recent = loss_log_b[-50:]
            avg_total = np.mean([x['total'] for x in recent])
            avg_s2tt = np.mean([x['s2tt'] for x in recent])
            avg_s2st = np.mean([x['s2st'] for x in recent])
            kd_ratio = np.mean([x['kd'] for x in recent])
            cur_lr = optimizer_b.param_groups[0]['lr']
            print(f'  [B] Step {step+1:>4}/{STAGE_B_STEPS} | '
                  f'loss={avg_total:.4f} (s2tt={avg_s2tt:.4f} s2st={avg_s2st:.4f}) | '
                  f'KD={kd_ratio:.0%} | lr={cur_lr:.2e}')
        
        if (step + 1) % 500 == 0:
            save_checkpoint({
                'stage': 'B', 'step': step + 1,
                'dec_state': model_student.text_decoder.state_dict(),
                't2u_enc_state': model_student.t2u_model.model.encoder.state_dict(),
                't2u_dec_state': model_student.t2u_model.model.decoder.state_dict(),
                'optimizer_state': optimizer_b.state_dict(),
                'loss_log': loss_log_b,
            }, 'phase6_stage_b', step + 1)
            
            # Checkpoint eval
            print(f'\n  Checkpoint eval at step {step+1}...')
            ckpt_chrf = quick_eval_chrf(model_student, eval_samples, max_samples=16)
            print(f'  ChrF: {ckpt_chrf:.2f}\n')
    
    except Exception as e:
        print(f'  [B] Step {step+1} error: {e}')
        continue

print(f'\n✓ Stage B complete. Avg loss: {np.mean([x["total"] for x in loss_log_b[-100:]]):.4f}')



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE C: Full Joint Training (speech encoder + decoder + T2U, 1 epoch)
# ══════════════════════════════════════════════════════════════════════════════

print('\n[STAGE C] Full Joint Training (all components, final polish)')

# Apply LoRA to speech encoder
model_student.speech_encoder = get_peft_model(
    model_student.speech_encoder, lora_config_encoder)
model_student.speech_encoder.print_trainable_parameters()

# Stage C hyperparameters
STAGE_C_STEPS = 1000
BATCH_ACCUM_C = 8
LR_C = 5e-5

optimizer_c = torch.optim.AdamW([
    {'params': [p for p in model_student.speech_encoder.parameters() if p.requires_grad],
     'lr': LR_C, 'weight_decay': 0.01},
    {'params': [p for p in model_student.text_decoder.parameters() if p.requires_grad],
     'lr': LR_C, 'weight_decay': 0.01},
    {'params': [p for p in model_student.t2u_model.parameters() if p.requires_grad],
     'lr': LR_C, 'weight_decay': 0.01},
], betas=(0.9, 0.98))

warmup_steps_c = int(0.1 * STAGE_C_STEPS)
def get_lr_scale_c(step):
    if step < warmup_steps_c:
        return step / warmup_steps_c
    return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps_c) / (STAGE_C_STEPS - warmup_steps_c)))

scheduler_c = torch.optim.lr_scheduler.LambdaLR(optimizer_c, get_lr_scale_c)
scaler_c = torch.cuda.amp.GradScaler()

# Language-balanced sampling for Stage C
lang_samples = defaultdict(list)
for s in ft_samples:
    lang_samples[s['tgt_lang']].append(s)

print(f'Language distribution:')
for lang, samples in lang_samples.items():
    print(f'  {lang}: {len(samples)} samples')

loss_log_c = []
print(f'\nTraining Stage C: {STAGE_C_STEPS} steps, LR={LR_C:.2e}')

for step in range(STAGE_C_STEPS):
    # Language-balanced sampling
    tgt_lang = random.choice(list(lang_samples.keys()))
    sample = random.choice(lang_samples[tgt_lang])
    
    try:
        inputs = processor(audio=sample['wav'], sampling_rate=16000, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Mix KD and supervised (60% supervised in final stage)
        use_kd = random.random() < 0.4
        
        if use_kd:
            with torch.no_grad():
                teacher_out = model_teacher.generate(
                    **inputs, tgt_lang=sample['tgt_lang'],
                    return_intermediate_token_ids=True)
                labels = teacher_out.sequences.to(device)
        else:
            text_inputs = processor.tokenizer(
                sample['ref'], return_tensors='pt', padding=True, truncation=True)
            labels = text_inputs['input_ids'].to(device)
        
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model_student(
                **inputs, tgt_lang=sample['tgt_lang'], labels=labels)
            
            s2tt_loss = outputs.loss if hasattr(outputs, 'loss') else torch.tensor(0.0).to(device)
            s2st_loss = outputs.t2u_loss if hasattr(outputs, 't2u_loss') else torch.tensor(0.0).to(device)
            
            # Balanced loss for final stage
            loss = 0.60 * s2tt_loss + 0.40 * s2st_loss
        
        scaler_c.scale(loss / BATCH_ACCUM_C).backward()
        loss_log_c.append({
            'total': loss.item(),
            's2tt': s2tt_loss.item(),
            's2st': s2st_loss.item(),
            'lang': tgt_lang
        })
        
        if (step + 1) % BATCH_ACCUM_C == 0:
            scaler_c.unscale_(optimizer_c)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.parameters() if p.requires_grad], 1.0)
            scaler_c.step(optimizer_c)
            scaler_c.update()
            optimizer_c.zero_grad()
            scheduler_c.step()
            torch.cuda.empty_cache()
        
        if (step + 1) % 50 == 0:
            recent = loss_log_c[-50:]
            avg_total = np.mean([x['total'] for x in recent])
            avg_s2tt = np.mean([x['s2tt'] for x in recent])
            avg_s2st = np.mean([x['s2st'] for x in recent])
            cur_lr = optimizer_c.param_groups[0]['lr']
            print(f'  [C] Step {step+1:>4}/{STAGE_C_STEPS} | '
                  f'loss={avg_total:.4f} (s2tt={avg_s2tt:.4f} s2st={avg_s2st:.4f}) | '
                  f'lr={cur_lr:.2e}')
        
        if (step + 1) % 500 == 0:
            save_checkpoint({
                'stage': 'C', 'step': step + 1,
                'enc_state': model_student.speech_encoder.state_dict(),
                'dec_state': model_student.text_decoder.state_dict(),
                't2u_enc_state': model_student.t2u_model.model.encoder.state_dict(),
                't2u_dec_state': model_student.t2u_model.model.decoder.state_dict(),
                'optimizer_state': optimizer_c.state_dict(),
                'loss_log': loss_log_c,
            }, 'phase6_stage_c', step + 1)
            
            print(f'\n  Checkpoint eval at step {step+1}...')
            ckpt_chrf = quick_eval_chrf(model_student, eval_samples, max_samples=16)
            print(f'  ChrF: {ckpt_chrf:.2f}\n')
    
    except Exception as e:
        print(f'  [C] Step {step+1} error: {e}')
        continue

print(f'\n✓ Stage C complete. Avg loss: {np.mean([x["total"] for x in loss_log_c[-100:]]):.4f}')



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Merge LoRA Adapters and Save
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '='*80)
print('  Merging LoRA adapters into base model...')
print('='*80)

model_student.speech_encoder = model_student.speech_encoder.merge_and_unload()
model_student.text_decoder = model_student.text_decoder.merge_and_unload()
model_student.t2u_model.model.encoder = model_student.t2u_model.model.encoder.merge_and_unload()
model_student.t2u_model.model.decoder = model_student.t2u_model.model.decoder.merge_and_unload()

model_student.eval()
sync_model_config(model_student)
gc.collect()
torch.cuda.empty_cache()

save_model_to_drive(model_student, processor, 'phase6_recovered_merged')
print('✓ Phase 6 complete. Model saved.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Training Visualization
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Phase 6: Multi-Stage Recovery Training', fontsize=14, fontweight='bold')

# Stage A loss
ax1 = axes[0, 0]
if loss_log_a:
    ema_a = []
    v = loss_log_a[0]
    for l in loss_log_a:
        v = 0.05*l + 0.95*v
        ema_a.append(v)
    ax1.plot(loss_log_a, alpha=0.2, color='#FF5722', lw=0.5, label='Raw')
    ax1.plot(ema_a, color='#FF5722', lw=2, label='EMA')
    ax1.set_title('Stage A: T2U Recovery', fontweight='bold')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('T2U Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)

# Stage B losses
ax2 = axes[0, 1]
if loss_log_b:
    steps_b = range(len(loss_log_b))
    s2tt_b = [x['s2tt'] for x in loss_log_b]
    s2st_b = [x['s2st'] for x in loss_log_b]
    
    # EMA smoothing
    def ema_smooth(data, alpha=0.05):
        ema = [data[0]]
        for x in data[1:]:
            ema.append(alpha*x + (1-alpha)*ema[-1])
        return ema
    
    ax2.plot(steps_b, ema_smooth(s2tt_b), color='#2196F3', lw=2, label='S2TT Loss')
    ax2.plot(steps_b, ema_smooth(s2st_b), color='#4CAF50', lw=2, label='S2ST Loss')
    ax2.set_title('Stage B: Joint Training', fontweight='bold')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(alpha=0.3)

# Stage C losses
ax3 = axes[1, 0]
if loss_log_c:
    steps_c = range(len(loss_log_c))
    total_c = [x['total'] for x in loss_log_c]
    ax3.plot(steps_c, ema_smooth(total_c), color='#9C27B0', lw=2, label='Total Loss')
    ax3.set_title('Stage C: Full Joint Training', fontweight='bold')
    ax3.set_xlabel('Step')
    ax3.set_ylabel('Loss')
    ax3.legend()
    ax3.grid(alpha=0.3)

# Language distribution in Stage C
ax4 = axes[1, 1]
if loss_log_c:
    lang_counts = defaultdict(int)
    for x in loss_log_c:
        lang_counts[x['lang']] += 1
    langs = list(lang_counts.keys())
    counts = [lang_counts[l] for l in langs]
    ax4.bar(langs, counts, color='#FF9800', alpha=0.85, edgecolor='white')
    ax4.set_title('Stage C: Language Balance', fontweight='bold')
    ax4.set_xlabel('Language')
    ax4.set_ylabel('Training Steps')
    ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
save_figure(fig, 'phase6_multistage_training.png')
plt.show()

print('\n✓ Training visualization saved.')

In [ ]:
# heed

In [ ]:
p6_bench = load_latest_checkpoint('phase6_benchmark')
if p6_bench:
    p6_results = p6_bench['results']
    p6_summary = p6_bench['summary']
    p6_detailed = p6_bench.get('detailed_summary')
    if not p6_detailed:
        p6_detailed = compute_detailed_summary(p6_results, 'P6_DoRA', p6_summary['params_M'])
else:
    p6_results, p6_summary = run_benchmark_asr(
        model_p6, eval_samples, 'P6_DoRA', save_n=4)
    p6_detailed = compute_detailed_summary(p6_results, 'P6_DoRA', p6_summary['params_M'])
    save_checkpoint(dict(
        results=p6_results,
        summary=p6_summary,
        detailed_summary=p6_detailed
    ), 'phase6_benchmark', 0)

store_summary(p6_summary)
store_detailed_summary(p6_detailed)
print_detailed_summary_table('P6_DoRA')
plot_phase_comparison()
plot_detailed_phase_comparison()

if loss_log_p6:
    fig, ax = plt.subplots(figsize=(10, 5))
    ema, v = [], loss_log_p6[0]
    for l in loss_log_p6:
        v = 0.05*l + 0.95*v
        ema.append(v)
    ax.plot(loss_log_p6, alpha=0.2, color='#FF5722', lw=0.5, label='Raw')
    ax.plot(ema, color='#FF5722', lw=2, label='EMA')
    ax.set_title('Phase 6: DoRA Fine-tuning Loss', fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, 'phase6_dora_training.png')
    plt.show()

del model_p5
gc.collect()
torch.cuda.empty_cache()
print('P5 model freed.')

---
## Phase 7: Textless Inference + Full Comprehensive Benchmark
Evaluation of the final ~673M textless model:
1. Translation quality — ASR-ChrF, all 5 languages, bidirectional
2. Voice cloning — ECAPA cosine similarity (input vs output speaker)
3. Long-form audio — 5s, 15s, 30s, 60s (chunked inference for >25s)
4. Audio quality — UTMOS naturalness score
5. Speed — RTF comparison vs V1 and teacher


In [ ]:
# ── Load final model ──────────────────────────────────────────────────────────
model_final, processor = load_model_from_drive('phase6_dora_merged')

# Rebuild CIF + speaker adapter on model_final if needed
if not hasattr(model_final,'cif_connector') or model_final.cif_connector is None:
    model_final.cif_connector  = CIFConnector(d_model=hidden, n_langs=n_langs+5)
    model_final.speaker_adapter = SpeakerAdapter()

p6b_final = load_latest_checkpoint('phase6b_e2e')
if p6b_final:
    model_final.cif_connector.load_state_dict(p6b_final.get('cif_connector', {}), strict=False)
    model_final.speaker_adapter.load_state_dict(p6b_final.get('speaker_adapter', {}), strict=False)
    print('Final CIF + speaker adapter weights loaded.')

model_final.eval()
model_final = _consolidate_to_single_gpu(model_final)
device_final = torch.device('cuda:0')
print_model_breakdown(model_final, 'FINAL ~673M Textless Model')
gpu_mem()


In [ ]:

def translate_longform(mdl, audio_wav, tgt_lang, chunk_s=25, overlap_s=2, sr=16000):
    chunk_len   = chunk_s * sr
    overlap_len = overlap_s * sr
    hop_len     = chunk_len - overlap_len
    chunks, pos = [], 0
    while pos < len(audio_wav):
        chunk = audio_wav[pos : pos + chunk_len]
        if len(chunk) < sr // 2:
            break
        chunks.append(chunk)
        pos += hop_len
    print(f'Long-form {len(audio_wav)/sr:.1f}s → {len(chunks)} chunk(s) × {chunk_s}s')
    outputs = []
    for i, chunk in enumerate(chunks):
        wav_out, rtf, _ = run_s2st(mdl, chunk, tgt_lang)
        if i > 0 and len(wav_out) > overlap_len // 2:
            wav_out = wav_out[overlap_len // 2:]
        outputs.append(wav_out)
        print(f'  Chunk {i+1}/{len(chunks)} RTF={rtf:.3f}')
    return np.concatenate(outputs) if outputs else np.zeros(sr)

print('✓ Textless inference ready.')

In [ ]:
eval_samples[0]

DEBUG T2U

In [ ]:
# ── DEMO: Listen to voice-cloned translation ──────────────────────────────────
demo = eval_samples[0]
print(f'Source (EN): {demo["src_text"]}')
play(demo['wav'], 16000, 'Input (English)')

tgt = 'ben'
print(f'\nTranslating EN→{tgt.upper()}...')
try:
    wav_out, rtf, _ = run_s2st(model_final, demo['wav'], tgt_lang=tgt)
    
    hyp = asr_transcribe(wav_out, tgt)
    print(f'  ASR: {hyp[:120]}')
    print(f'  RTF: {rtf:.3f}')
    play(wav_out, 16000, f'Output ({tgt}, voice-cloned)')
    save_audio(wav_out, 16000, f'demo_{tgt}.wav')
except Exception as e:
    print(f'  Error: {e}')


In [ ]:
# ── BENCHMARK 1: Translation quality — all 5 languages, bidirectional ─────────
p7_trans_ckpt = load_latest_checkpoint('phase7_translation')
if p7_trans_ckpt:
    trans_results = p7_trans_ckpt['results']
    print('Loaded translation results.')
else:
    trans_results = {}
    model_final.eval()
    
    # Group eval_samples by language pair
    from collections import defaultdict
    samples_by_pair = defaultdict(list)
    for s in eval_samples:
        pair_key = f"{s['src_lang']}→{s['tgt_lang']}"
        samples_by_pair[pair_key].append(s)
    
    for pair_key, pair_samples in samples_by_pair.items():
        print(f'\nBenchmarking {pair_key} ({len(pair_samples)} samples)...')
        pair_res = []
        for s in pair_samples:
            try:
                wav_out, rtf, _ = run_s2st(model_final, s['wav'], tgt_lang=s['tgt_lang'])
                hyp  = asr_transcribe(wav_out, s['tgt_lang'])
                chrf = compute_chrf(hyp, s['ref'])
                bleu = compute_bleu(hyp, s['ref'])
                pair_res.append(dict(id=s['id'],hyp=hyp,ref=s['ref'],chrf=chrf,bleu=bleu,rtf=rtf))
            except Exception as e:
                print(f'  Error: {e}')
                pair_res.append(dict(id=s.get('id','?'),hyp='',ref=s.get('ref',''),chrf=0,bleu=0,rtf=0))
        
        trans_results[pair_key] = dict(
            results=pair_res,
            avg_chrf=float(np.mean([r['chrf'] for r in pair_res])),
            avg_bleu=float(np.mean([r['bleu'] for r in pair_res])),
            avg_rtf =float(np.mean([r['rtf']  for r in pair_res])),
        )
        print(f'  {pair_key}: ASR-ChrF={trans_results[pair_key]["avg_chrf"]:.2f} '
              f'ASR-BLEU={trans_results[pair_key]["avg_bleu"]:.2f} RTF={trans_results[pair_key]["avg_rtf"]:.4f}')
    
    save_checkpoint({'results': trans_results}, 'phase7_translation', 0)

print('\n--- Translation Quality (ASR-ChrF/BLEU) ---')
print(f'  {"Pair":<15} {"ASR-ChrF":>10} {"ASR-BLEU":>10} {"RTF":>7}')
for pair, res in trans_results.items():
    print(f'  {pair:<15} {res["avg_chrf"]:>10.2f} {res["avg_bleu"]:>10.2f} {res["avg_rtf"]:>7.4f}')


In [ ]:
# ── BENCHMARK 2: Voice cloning — ECAPA speaker similarity ────────────────────
p7_spk_ckpt = load_latest_checkpoint('phase7_speaker_sim')
if p7_spk_ckpt:
    spk_results = p7_spk_ckpt['results']; print('Loaded speaker sim results.')
else:
    spk_results = []
    # Test on subset of language pairs
    test_pairs = [('eng','ben'),('eng','hin'),('eng','cmn'),('eng','arb')]
    for src_lang, tgt_lang in test_pairs:
        pair_samples = [s for s in eval_samples if s['src_lang']==src_lang and s['tgt_lang']==tgt_lang][:10]
        print(f'  Speaker sim {src_lang}→{tgt_lang}...')
        for s in pair_samples:
            try:
                wav_out, rtf, _ = run_s2st(model_final, s['wav'], tgt_lang=tgt_lang)
                src_emb = extract_speaker_emb(s['wav'])
                out_emb = extract_speaker_emb(wav_out) if len(wav_out)>800 else src_emb*0
                sim = F.cosine_similarity(src_emb.unsqueeze(0), out_emb.unsqueeze(0)).item()
                spk_results.append({'id':s['id'],'pair':f'{src_lang}→{tgt_lang}',
                                    'speaker_sim':sim,'rtf':rtf})
                print(f'    {s["id"]}: sim={sim:.3f}')
            except Exception as e:
                print(f'    Error: {e}')
    save_checkpoint({'results': spk_results}, 'phase7_speaker_sim', 0)

if spk_results:
    avg_sim = np.mean([r['speaker_sim'] for r in spk_results])
    qual = ('Excellent' if avg_sim>0.85 else 'Good' if avg_sim>0.70
            else 'Acceptable' if avg_sim>0.55 else 'Poor')
    print(f'\nVoice cloning — avg ECAPA sim: {avg_sim:.3f}  [{qual}]')
    print(f'  Target: 0.65–0.78  |  SeamlessExpressive: ~0.80')


In [ ]:
# ── BENCHMARK 3: Long-form audio (PLAN.md Section 2.3) ───────────────────────
p7_lf_ckpt = load_latest_checkpoint('phase7_longform')
if p7_lf_ckpt:
    longform_results = p7_lf_ckpt['results']; print('Loaded long-form results.')
else:
    longform_results = {}
    DURATIONS = [5, 15, 30, 60]
    base_wavs = [s['wav'] for s in eval_samples[:8]]
    base_refs = [s['ref'] for s in eval_samples[:8]]

    def make_test_audio(target_s, wavs, sr=16000):
        combined = np.concatenate(wavs)
        tlen = target_s * sr
        if len(combined) < tlen:
            reps = math.ceil(tlen/len(combined))
            combined = np.tile(combined, reps)
        return combined[:tlen]

    model_final.eval()
    for dur_s in DURATIONS:
        print(f'\nLong-form {dur_s}s...')
        test_wav = make_test_audio(dur_s, base_wavs)
        test_ref = ' '.join(base_refs)
        chrfs, rtfs = [], []
        for trial in range(3):
            try:
                if dur_s <= 25:
                    wav_out, rtf, _ = run_s2st(model_final, test_wav, tgt_lang='ben')
                else:
                    t0 = time.time()
                    wav_out = translate_longform(model_final, test_wav, tgt_lang='ben')
                    rtf = (time.time()-t0)/dur_s
                if len(wav_out)>800:
                    hyp = asr_transcribe(wav_out, 'ben')
                    chrfs.append(compute_chrf(hyp, test_ref[:300]))
                    rtfs.append(rtf)
            except Exception as e:
                print(f'  Trial {trial+1} error: {e}')
        longform_results[dur_s] = {
            'duration_s': dur_s, 'method': 'direct' if dur_s<=25 else 'chunked_25s+2s_overlap',
            'avg_chrf': float(np.mean(chrfs)) if chrfs else 0,
            'avg_rtf':  float(np.mean(rtfs))  if rtfs  else 0,
        }
        print(f'  {dur_s}s: ChrF={longform_results[dur_s]["avg_chrf"]:.2f} RTF={longform_results[dur_s]["avg_rtf"]:.3f}')
    save_checkpoint({'results': longform_results}, 'phase7_longform', 0)

print('\nLong-form results:')
for dur, res in sorted(longform_results.items()):
    print(f'  {dur}s [{res["method"]}]: ChrF={res["avg_chrf"]:.2f}  RTF={res["avg_rtf"]:.3f}')


In [ ]:
# ── FINAL COMPREHENSIVE VISUALISATION (paper figures) ─────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle('Textless SeamlessM4T v2 (~673M): Comprehensive Benchmark - 5 Languages',
             fontsize=14, fontweight='bold', y=0.99)

# 1: Parameter evolution
ax1 = fig.add_subplot(3,3,1)
phase_names  = ['Teacher\n1805M','V1\n1039M','Vocab5L\n824M','Enc16L\n630M',
                'LaCoT2U\n542M','Textless\n673M']
phase_params = [1805, 1039, 824, 630, 542, 673]
colors_pb = ['#9E9E9E']*5 + ['#4CAF50']
bars = ax1.bar(range(len(phase_names)), phase_params, color=colors_pb, alpha=0.85, edgecolor='white')
bars[-1].set_edgecolor('#2E7D32'); bars[-1].set_linewidth(2)
ax1.set_xticks(range(len(phase_names))); ax1.set_xticklabels(phase_names, fontsize=7)
ax1.set_ylabel('Parameters (M)'); ax1.set_title('Model Size Evolution', fontweight='bold')
for bar, v in zip(bars, phase_params):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, f'{v}M',
             ha='center', va='bottom', fontsize=7, fontweight='bold')

# 2: ASR-ChrF by lang pair (all 8 pairs)
ax2 = fig.add_subplot(3,3,2)
if trans_results:
    pairs_  = list(trans_results.keys())
    chrfs_  = [trans_results[p]['avg_chrf'] for p in pairs_]
    bleus_  = [trans_results[p]['avg_bleu'] for p in pairs_]
    x_  = np.arange(len(pairs_)); w_ = 0.35
    ax2.bar(x_-w_/2, chrfs_, w_, label='ASR-ChrF', color='#2196F3', alpha=0.85)
    ax2.bar(x_+w_/2, bleus_, w_, label='ASR-BLEU', color='#FF9800', alpha=0.85)
    ax2.set_xticks(x_); ax2.set_xticklabels(pairs_, rotation=45, ha='right', fontsize=7)
    ax2.set_title('Translation Quality by Language Pair (ASR)', fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.axhline(35, color='green', ls=':', lw=1.5, alpha=0.7, label='Target')

# 3: Speaker similarity
ax3 = fig.add_subplot(3,3,3)
if spk_results:
    sims_spk = [r['speaker_sim'] for r in spk_results]
    ax3.hist(sims_spk, bins=12, color='#E91E63', alpha=0.8, edgecolor='white')
    for thresh, lbl, col in [(0.85,'Excellent','green'),(0.70,'Good','orange'),(0.55,'Acceptable','red')]:
        ax3.axvline(thresh, color=col, ls='--', lw=1.5, label=f'{lbl}>{thresh}')
    ax3.axvline(np.mean(sims_spk), color='black', ls='-', lw=2,
                label=f'Mean={np.mean(sims_spk):.3f}')
    ax3.set_xlabel('ECAPA Cosine Similarity'); ax3.set_title('Speaker Similarity (Voice Cloning)', fontweight='bold')
    ax3.legend(fontsize=7)

# 4: Long-form quality
ax4 = fig.add_subplot(3,3,4)
if longform_results:
    durs_ = sorted(longform_results.keys())
    lf_ch = [longform_results[d]['avg_chrf'] for d in durs_]
    lf_rt = [longform_results[d]['avg_rtf']  for d in durs_]
    ax4_t = ax4.twinx()
    ax4.plot(durs_, lf_ch, 'o-', color='#4CAF50', lw=2, ms=8, label='ASR-ChrF')
    ax4_t.plot(durs_, lf_rt, 's--', color='#FF5722', lw=2, ms=8, label='RTF')
    ax4.axvline(25, color='gray', ls=':', lw=1.5, label='Chunking boundary')
    ax4.set_xlabel('Duration (s)'); ax4.set_ylabel('ASR-ChrF', color='#4CAF50')
    ax4_t.set_ylabel('RTF', color='#FF5722')
    ax4.set_title('Long-Form: Quality vs Duration', fontweight='bold')
    ax4.legend(loc='upper left', fontsize=8); ax4_t.legend(loc='upper right', fontsize=8)

# 5: RTF comparison
ax5 = fig.add_subplot(3,3,5)
final_rtf = np.mean([v['avg_rtf'] for v in trans_results.values()]) if trans_results else 0.09
spd_labels = ['Teacher\n1805M','V1\n1039M','Textless\n673M']
spd_rtfs   = [0.268, 0.113, final_rtf]
ax5.bar(spd_labels, spd_rtfs, color=['#F44336','#FF9800','#4CAF50'], alpha=0.85, edgecolor='white')
ax5.set_ylabel('RTF (lower=faster)'); ax5.set_title('Inference Speed (RTF)', fontweight='bold')
for i,(l,v) in enumerate(zip(spd_labels,spd_rtfs)):
    ax5.text(i, v+0.003, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# 6: Speaker sim by pair
ax6 = fig.add_subplot(3,3,6)
if spk_results:
    from collections import defaultdict
    pair_sims_ = defaultdict(list)
    for r in spk_results: pair_sims_[r['pair']].append(r['speaker_sim'])
    pn = list(pair_sims_.keys())
    pm = [np.mean(pair_sims_[p]) for p in pn]
    ps = [np.std(pair_sims_[p]) for p in pn]
    ax6.bar(pn, pm, yerr=ps, capsize=5, color='#9C27B0', alpha=0.8, edgecolor='white')
    ax6.axhline(0.65, color='green', ls='--', lw=1.5, label='Target 0.65')
    ax6.set_ylim(0,1); ax6.set_ylabel('Speaker Similarity')
    ax6.set_xticklabels(pn, rotation=30, ha='right', fontsize=7)
    ax6.set_title('Speaker Sim by Language Pair', fontweight='bold'); ax6.legend(fontsize=8)

# 7: Enc pruning ChrF curve (Phase 2)
ax7 = fig.add_subplot(3,3,7)
if 'p2_log' in dir() and p2_log:
    iters7 = [e['iter'] for e in p2_log]; chrfs7 = [e['chrf'] for e in p2_log]
    ax7.plot(iters7, chrfs7, 'o-', color='#FF9800', lw=2, ms=7)
    for e in p2_log:
        ax7.annotate(f'L{e["removed"]}', (e['iter'],e['chrf']),
                     fontsize=6, ha='center', va='bottom')
    ax7.set_xlabel('Pruning iter'); ax7.set_ylabel('ASR-ChrF')
    ax7.set_title('Enc Pruning: ASR-ChrF per Removal', fontweight='bold')
else:
    ax7.text(0.5,0.5,'P2 log not in session', ha='center', va='center', transform=ax7.transAxes)

# 8: Per-language-pair scatter
ax8 = fig.add_subplot(3,3,8)
if trans_results:
    all_chrfs = []
    all_bleus = []
    for pair_key, pair_data in trans_results.items():
        for r in pair_data['results']:
            all_chrfs.append(r['chrf'])
            all_bleus.append(r['bleu'])
    if all_chrfs:
        ax8.scatter(all_bleus, all_chrfs, color='#2196F3', alpha=0.5, s=30, edgecolors='white')
        mu_c = np.mean(all_chrfs)
        ax8.axhline(mu_c, color='red', ls='--', lw=1.5, label=f'Mean ChrF={mu_c:.1f}')
        ax8.set_xlabel('ASR-BLEU'); ax8.set_ylabel('ASR-ChrF')
        ax8.set_title('All Pairs: BLEU vs ChrF per sample', fontweight='bold'); ax8.legend(fontsize=8)

# 9: Architecture comparison table
ax9 = fig.add_subplot(3,3,9)
ax9.axis('off')
tbl_data = [
    ['Component','Original','Textless 673M'],
    ['Text Decoder','867M 24L','0M (removed)'],
    ['lm_head+vocab','~262M','0M (removed)'],
    ['Speech Encoder','635M 24L','~441M 16L'],
    ['T2U Model','262M 6+6L','~175M 4+4L'],
    ['CIF Connector','—','~5M (NEW)'],
    ['Speaker Adapter','—','~0.1M (NEW)'],
    ['Vocoder','41.9M','41.9M (frozen)'],
    ['TOTAL','1805M','~673M'],
]
tbl = ax9.table(cellText=tbl_data[1:], colLabels=tbl_data[0],
                cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.2, 1.5)
for j in range(3):
    tbl[len(tbl_data)-1, j].set_facecolor('#C8E6C9')
    tbl[len(tbl_data)-1, j].set_text_props(fontweight='bold')
tbl[1,2].set_facecolor('#FFCDD2'); tbl[2,2].set_facecolor('#FFCDD2')
ax9.set_title('Architecture Comparison', fontweight='bold', pad=10)

plt.tight_layout(rect=[0,0,1,0.98])
save_figure(fig, 'phase7_comprehensive_benchmark.png')
plt.show()
print('✓ Comprehensive benchmark figure saved (5 languages, ASR metrics).')


In [ ]:
# ── FINAL PAPER TABLE ─────────────────────────────────────────────────────────
print('\n' + '='*80)
print('  FINAL RESULTS — Textless SeamlessM4T v2 ~673M')
print('  Target: INTERSPEECH 2026 · IWSLT 2026 Cross-Lingual Voice Cloning Track')
print('='*80)

avg_chrf_final = np.mean([v['avg_chrf'] for v in trans_results.values()]) if trans_results else 0
avg_bleu_final = np.mean([v['avg_bleu'] for v in trans_results.values()]) if trans_results else 0

print('\n[Table 1: Parameter Reduction]')
print(f'  Teacher (1805M) → V1 (1039M) → Textless (673M)')
print(f'  Compression from teacher: {(1-673/1805)*100:.1f}%')
print(f'  Compression from V1:      {(1-673/1039)*100:.1f}%')

print('\n[Table 2: Translation Quality - All Language Pairs]')
print(f'  {"Pair":<15} {"ASR-ChrF":>10} {"ASR-BLEU":>10} {"RTF":>8}')
for pair, res in sorted(trans_results.items()):
    print(f'  {pair:<15} {res["avg_chrf"]:>10.2f} {res["avg_bleu"]:>10.2f} {res["avg_rtf"]:>8.4f}')
print(f'  {"Average":<15} {avg_chrf_final:>10.2f} {avg_bleu_final:>10.2f}')

print('\n[Table 3: Voice Cloning]')
if spk_results:
    avg_sim = np.mean([r['speaker_sim'] for r in spk_results])
    qual = 'Excellent' if avg_sim>0.85 else 'Good' if avg_sim>0.70 else 'Acceptable' if avg_sim>0.55 else 'Poor'
    print(f'  ECAPA Speaker Similarity: {avg_sim:.3f}  [{qual}]')
    print(f'  Target: 0.65–0.78  (SeamlessExpressive: ~0.80)')

print('\n[Table 4: Speed]')
final_rtf = np.mean([v['avg_rtf'] for v in trans_results.values()]) if trans_results else 0.09
print(f'  Teacher RTF: 0.268 | V1 RTF: 0.113 | Textless RTF: {final_rtf:.3f}')
if final_rtf > 0:
    print(f'  Speedup vs teacher: {0.268/final_rtf:.1f}×')

print('\n[Table 5: Long-Form Support]')
for dur, res in sorted(longform_results.items()):
    print(f'  {dur}s [{res["method"]}]: ASR-ChrF={res["avg_chrf"]:.2f}  RTF={res["avg_rtf"]:.3f}')

print('\n' + '='*80)

# Store final summary
final_summary = dict(
    label='P_Final_Textless_673M',
    params_M=673.0,
    avg_bleu=avg_bleu_final,
    avg_chrf=avg_chrf_final,
    avg_rtf=final_rtf,
    speaker_sim=np.mean([r['speaker_sim'] for r in spk_results]) if spk_results else 0,
    n=sum(len(v['results']) for v in trans_results.values()),
)
store_summary(final_summary)
plot_phase_comparison()
plot_size_vs_quality()

In [ ]:
# Upload all artefacts
if ON_KAGGLE:
    subprocess.run(f'rclone copy "{AUDIO_DIR}/" "{GDRIVE_ROOT}/audio/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M', shell=True)
    subprocess.run(f'rclone copy "{FIG_DIR}/" "{GDRIVE_ROOT}/figures/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M', shell=True)
    print('[rclone] Audio + figures synced to Drive.')

session_status()
print('\n✓ Phase 7 complete. All results persisted to Drive.')